# TAR-KG → Knowledge Graph (KG) Builder



## Relation types processed

| # | Source file | Output Relation | Head ID | Tail ID |
|---|---|---|---|---|
| 1 | Disease_Gene | Disease_Gene | DOID/MESH | NCBI Symbol |
| 2 | Disease_Anatomy | Disease_Anatomy | DOID/MESH | UBERON |
| 3 | Gene_Disease | Gene_Disease | NCBI Symbol | DOID/MESH |
| 4 | Disease_Compound | Disease_ChemicalEntity | DOID/MESH | PubChem CID |
| 5 | Gene_Phenotype | Gene_Phenotype | NCBI Symbol | HPO ID |
| 6 | Gene_Pathway | Gene_Pathway | NCBI Symbol | Reactome ID |
| 7 | Gene_Pathway (protein split) | Protein_Pathway | UniProt AC | Reactome ID |
| 8 | Gene_Anatomy | Gene_Anatomy | NCBI Symbol | UBERON |
| 9 | Go_Go | BiologicalProcess_BiologicalProcess etc. | GO ID | GO ID |
| 10 | Molecular_Function_Molecular_Function | MolecularFunction_MolecularFunction | GO ID | GO ID |
| 11 | Compound_Gene | ChemicalEntity_Gene | PubChem CID | NCBI Symbol |
| 12 | Anatomy_Gene | Anatomy_Gene | UBERON | NCBI Symbol |
| 13 | Phenotype_Phenotype | Phenotype_Phenotype | HPO ID | HPO ID |
| 14 | Pathway_Go | Pathway_GO | Reactome ID | GO ID |
| 15 | Gene_MolecularFunction | Gene_MolecularFunction | NCBI Symbol | GO ID |
| 16 | Gene_Go | Protein_BiologicalProcess / Protein_CellularComponent / Protein_MolecularFunction | UniProt AC | GO ID |
| 17 | Cellular_Component_Cellular_Component | CellularComponent_CellularComponent | GO ID | GO ID |
| 18 | Gene_Compound | Gene_ChemicalEntity | NCBI Symbol | PubChem CID |
| 19 | Gene_Gene (numeric head) | Gene_Gene | NCBI Symbol | NCBI Symbol |
| 20 | Gene_Gene (non-numeric head) | Protein_Protein | UniProt AC | UniProt AC |
| 21 | Disease_Disease | Disease_Disease | DOID/MESH | DOID/MESH |
| 22 | Compound_Disease | ChemicalEntity_Disease | PubChem CID | DOID |
| 23 | Gene_BiologicalProcess | Gene_BiologicalProcess | NCBI Symbol | GO ID |
| 24 | Compound_Phenotype | ChemicalEntity_Phenotype | PubChem CID | HPO ID |
| 25 | Compound_SideEffect | ChemicalEntity_SideEffect | PubChem CID | HPO ID |
| 26 | Anatomy_Anatomy | Anatomy_Anatomy | UBERON | UBERON |
| 27 | Pathway_Pathway | Pathway_Pathway | Reactome ID | Reactome ID |



---
## 0 · Configuration — edit ONLY these two lines

In [1]:
import os
import re
import glob
import numpy as np
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# USER CONFIGURATION
# BASE_PATH : root folder containing all raw input data
# OUT_PATH  : folder where all processed CSVs will be saved
# ─────────────────────────────────────────────────────────────────────────────
BASE_PATH = "/storage/Arushi/090526_EvoAge/kg_formation/data_collection/"
OUT_PATH  = "/storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/"

# ── Derived input paths ───────────────────────────────────────────────────────
PUBCHEM_PKL_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/combined_df.pkl")
PUBCHEM_DB_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/Pubchem_ID_DrugBank_Chebi.csv")
PUBCHEM_SYN_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/pubchem/CID-Synonym-filtered")
ENS2NCBI_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/ENSEMBL/ENSEMBLE_ID_2_NCBI_Gene_SYM.csv")
NCBI_HUMAN_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/ncbi/Homo_sapiens.gene_info")
OMIM_DO_PATH       = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/OMIMinDO.tsv")
MESH2DOID_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/DO/HumanDiseaseOntology-main/DOreports/MESHinDO.tsv")
MEDGEN_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGenIDMappings.txt")
MEDGEN_HPO_PATH    = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MeSH/MedGen_HPO_Mapping.txt")
MESH_COMB_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_Combined.csv")
MESH_SUPP_PATH     = os.path.join(BASE_PATH, "databases_for_mapping/MESH/Mesh_Supplementary_Info.csv")
DO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/DO/All_DO_ref_files.csv")
GO_PATH            = os.path.join(BASE_PATH, "databases_for_mapping/MESH/MESH_GO_ID_NAME.csv")
UBERON_PATH        = os.path.join(BASE_PATH, "databases_for_mapping/uberon/Uberon_ID_Name_non_dup.csv")
HPO_PATH           = os.path.join(BASE_PATH, "databases_for_mapping/hpo/HPO.csv")
REACTOME_PATH      = os.path.join(BASE_PATH, "databases_for_mapping/reactome/ReactomePathways.txt")
UNIPROT_REC_PATH   = os.path.join(BASE_PATH, "databases_for_mapping/uniprot/Uniprot_ID_Recname.csv")
TARKG_NODES_PATH   = os.path.join(BASE_PATH, "tarkg/TarKG_nodes.csv")
TARKG_NODEMAP_PATH = os.path.join(BASE_PATH, "tarkg/TarKG_nodes_mapping.csv")
TARKG_EDGES_PATH   = os.path.join(BASE_PATH, "tarkg/TarKG_edges.csv")
TARKG_EDGEMAP_PATH = os.path.join(BASE_PATH, "tarkg/TarKG_edges_mapping.csv")
TARKG_GENE_PATH    = os.path.join(BASE_PATH, "tarkg/Gene_nodes.csv")

# Per-relation intermediate CSVs are read from this folder
PROCESSED_TARKG    = os.path.join(BASE_PATH, "tarkg/Processed_TARKG/")

os.makedirs(OUT_PATH, exist_ok=True)

# Standard schema column order
DESIRED_COLS = [
    "Head", "Relation", "Tail", "Head_type", "Relation_type", "Tail_type", "Source",
    "KG_Source", "Head_Alt_ID", "Head_detail_name", "Tail_DO_Alt_ids",
    "Tail_IUPAC", "Tail_Smiles", "Tail_SMILES_name", "Tail_detail_name",
    "Head_ID_IS", "Tail_ID_IS", "Kg_index", "Kg", "Change"
]

print("Paths configured.")
print(f"  Input  root : {BASE_PATH}")
print(f"  Output dir  : {OUT_PATH}")

Paths configured.
  Input  root : /storage/Arushi/090526_EvoAge/kg_formation/data_collection/
  Output dir  : /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/


---
## Part 1 · Inspect raw TAR-KG files

In [96]:
#

In [3]:
# Load TAR-KG node and edge metadata files for inspection
Nfile = pd.read_csv(TARKG_NODES_PATH)
N_map = pd.read_csv(TARKG_NODEMAP_PATH)
E_file = pd.read_csv(TARKG_EDGES_PATH)
E_map  = pd.read_csv(TARKG_EDGEMAP_PATH)

print(f"Nodes: {len(Nfile):,} | Node mappings: {len(N_map):,}")
print(f"Edges: {len(E_file):,} | Edge mappings: {len(E_map):,}")
print("\nNode kind distribution:")
print(N_map['kind'].value_counts())
print("\nCompound db_source distribution:")
print(N_map[N_map['kind'] == 'Compound']['db_source'].value_counts())
print("\nEdge relation distribution:")
print(E_file['relation'].value_counts() if 'relation' in E_file.columns else E_file.dtypes)

/tmp/ipykernel_1739243/2115820059.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  E_map  = pd.read_csv(TARKG_EDGEMAP_PATH)


Nodes: 1,143,313 | Node mappings: 1,731,097
Edges: 32,806,467 | Edge mappings: 42,378,099

Node kind distribution:
kind
Compound              971582
Gene                  340827
Biological_Process    131315
Disease               103104
Molecular_Function     45383
Pathway                36467
Anatomy                30611
Phenotype              28696
Cellular_Component     17588
Side_Effect            15389
TCM_CMM                 3212
TCM_Prescription        2626
TCM_Symptom             2357
Symptom                 1708
TCM_Syndrome             232
Name: count, dtype: int64

Compound db_source distribution:
db_source
CHEMBL         779973
PUBCHEM         87521
DrugBank        37348
MESH            29048
INTEDE           3748
BRENDA            724
CHEBI             574
MOLPORT           221
BINDINGDB         135
ZINC               63
GTOPDB             51
DRUGCENTRAL        18
Name: count, dtype: int64

Edge relation distribution:
relation
associated with                  12707139
parti

---
## Part 1 · Build `Processed_TARKG/` from raw TAR-KG files

The raw TAR-KG download has these files:
```
TarKG_edges.csv          — columns: index, node1, node1_type, relation, node2, node2_type
TarKG_edges_mapping.csv  — same + db_source, kg_index, kg, change
TarKG_nodes.csv          — columns: index, unify_id, kind, dbid, db_source, name, source
TarKG_nodes_mapping.csv  — same + kgid, kg_index, kg
Gene_nodes.csv           — gene-specific node file
```

This section:
1. Inspects edge and node type distributions
2. Joins `TarKG_edges_mapping.csv` onto `TarKG_edges.csv` to enrich edges with `db_source`, `kg_index`, `kg`, `change`
3. Splits the enriched edge table by `(node1_type, node2_type)` pair and saves one CSV per relation type into `Processed_TARKG/`

These per-relation files are what all downstream processing cells read from.

In [4]:
# ── Inspect raw node and edge files ──────────────────────────────────────────
Nfile = pd.read_csv(TARKG_NODES_PATH)
N_map = pd.read_csv(TARKG_NODEMAP_PATH)
E_file = pd.read_csv(TARKG_EDGES_PATH)
E_map  = pd.read_csv(TARKG_EDGEMAP_PATH)

print(f"Nodes:         {len(Nfile):,} rows")
print(f"Node mappings: {len(N_map):,} rows")
print(f"Edges:         {len(E_file):,} rows")
print(f"Edge mappings: {len(E_map):,} rows")
print()
print("Node kind distribution:")
print(N_map['kind'].value_counts())
print()
print("Compound db_source distribution:")
print(N_map[N_map['kind'] == 'Compound']['db_source'].value_counts())

/tmp/ipykernel_1739243/2243910178.py:5: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  E_map  = pd.read_csv(TARKG_EDGEMAP_PATH)


Nodes:         1,143,313 rows
Node mappings: 1,731,097 rows
Edges:         32,806,467 rows
Edge mappings: 42,378,099 rows

Node kind distribution:
kind
Compound              971582
Gene                  340827
Biological_Process    131315
Disease               103104
Molecular_Function     45383
Pathway                36467
Anatomy                30611
Phenotype              28696
Cellular_Component     17588
Side_Effect            15389
TCM_CMM                 3212
TCM_Prescription        2626
TCM_Symptom             2357
Symptom                 1708
TCM_Syndrome             232
Name: count, dtype: int64

Compound db_source distribution:
db_source
CHEMBL         779973
PUBCHEM         87521
DrugBank        37348
MESH            29048
INTEDE           3748
BRENDA            724
CHEBI             574
MOLPORT           221
BINDINGDB         135
ZINC               63
GTOPDB             51
DRUGCENTRAL        18
Name: count, dtype: int64


In [5]:
# ── Edge relation type distribution ───────────────────────────────────────────
# Build the Relation column: node1_type_node2_type
E_file['Relation'] = E_file['node1_type'] + '_' + E_file['node2_type']
print("Edge relation distribution (node1_type_node2_type):")
print(E_file['Relation'].value_counts())
print()
print("Unique relation strings:", E_file['Relation'].nunique())

Edge relation distribution (node1_type_node2_type):
Relation
TCM_CMM_Gene                             5812852
Gene_Gene                                5690902
Compound_Gene                            4240151
Compound_Compound                        3615725
Anatomy_Gene                             3529632
Disease_Gene                             2219276
Compound_Disease                         1962151
TCM_CMM_Disease                          1612380
Gene_Biological_Process                   996493
TCM_Syndrome_Disease                      422383
Gene_Cellular_Component                   400070
TCM_Symptom_Disease                       378903
Gene_Molecular_Function                   346409
Gene_Pathway                              331951
Gene_Phenotype                            160692
Compound_Side_Effect                      148938
Gene_Disease                              145076
Disease_Phenotype                         130627
TCM_CMM_Compound                          124806
Compound

In [6]:
# ── Load Gene_nodes.csv — used to validate gene IDs ──────────────────────────
ALL_G = pd.read_csv(TARKG_GENE_PATH)
print(f"Gene nodes: {len(ALL_G):,} rows")
print("Columns:", list(ALL_G.columns))
print()
# Show which KGs contribute gene nodes
if 'kg' in ALL_G.columns:
    print("Gene node kg distribution:")
    print(ALL_G['kg'].value_counts())

Gene nodes: 340,827 rows
Columns: ['index', 'unify_id', 'kind', 'kgid', 'dbid', 'db_source', 'reviewed', 'name', 'organism', 'taxid', 'sequence', 'source', 'kg_index', 'kg']

Gene node kg distribution:
kg
BioKG          116238
addKG           97907
DRKG            32642
PrimeKG         18969
Hetionet        18952
OpenBioLink     18941
MSI             16503
tcmKG           15947
PharmKG          4728
Name: count, dtype: int64


In [7]:
# ── Join edge mapping onto edges to enrich with db_source, kg, change ────────
# TarKG_edges.csv:         index, node1, node1_type, relation, node2, node2_type
# TarKG_edges_mapping.csv: same + db_source, kg_index, kg, change
#
# The mapping file has multiple rows per edge index (one per source KG that
# contributed that edge). We deduplicate by edge index keeping the first
# occurrence (highest-priority source), then merge.

E_map_dedup = E_map.drop_duplicates(subset='index', keep='first')

E_enriched = E_file.merge(
    E_map_dedup[['index', 'db_source', 'kg_index', 'kg', 'change']],
    on='index',
    how='left'
)

# Standardise column names to match what processing cells expect
E_enriched = E_enriched.rename(columns={
    'node1':      'Head',
    'node2':      'Tail',
    'node1_type': 'Head_type',
    'node2_type': 'Tail_type',
    'relation':   'Relation_type',
    'Relation':   'Relation',
    'db_source':  'Source',
    'kg':         'Kg',
    'kg_index':   'Kg_index',
    'change':     'Change'
})

print(f"Enriched edge table: {len(E_enriched):,} rows")
print("Columns:", list(E_enriched.columns))
E_enriched.head(3)

KeyboardInterrupt: 

In [ ]:
# # ── Split by Relation and save one CSV per type ──────────────────────────────
# os.makedirs(PROCESSED_TARKG, exist_ok=True)

# # Normalise relation string -> filename
# # The raw data uses spaces and multi-word underscores (e.g. 'Compound_Side_Effect',
# # 'Gene_Biological_Process') while the downstream processing cells expect the
# # compressed forms ('Compound_SideEffect', 'Gene_BiologicalProcess').
# # This dict maps every observed relation string -> canonical filename.
# relation_to_filename = {
#     # ── Core relations (in both old and new download) ────────────────────────
#     'Disease_Gene':                              'Disease_Gene.csv',
#     'Disease_Disease':                           'Disease_Disease.csv',
#     'Disease_Anatomy':                           'Disease_Anatomy.csv',
#     'Disease_Compound':                          'Disease_Compound.csv',
#     'Disease_Pathway':                           'Disease_Pathway.csv',
#     'Disease_Phenotype':                         'Disease_Phenotype.csv',
#     'Disease_Symptom':                           'Disease_Symptom.csv',
#     'Gene_Disease':                              'Gene_Disease.csv',
#     'Gene_Gene':                                 'Gene_Gene.csv',
#     'Gene_Pathway':                              'Gene_Pathway.csv',
#     'Gene_Anatomy':                              'Gene_Anatomy.csv',
#     'Gene_Phenotype':                            'Gene_Phenotype.csv',
#     'Gene_Compound':                             'Gene_Compound.csv',
#     'Gene_Go':                                   'Gene_Go.csv',
#     'Gene_MolecularFunction':                    'Gene_MolecularFunction.csv',
#     # New download uses multi-word underscore variants:
#     'Gene_Molecular_Function':                   'Gene_MolecularFunction.csv',
#     'Gene_BiologicalProcess':                    'Gene_BiologicalProcess.csv',
#     'Gene_Biological_Process':                   'Gene_BiologicalProcess.csv',
#     'Gene_CellularComponent':                    'Gene_CellularComponent.csv',
#     'Gene_Cellular_Component':                   'Gene_CellularComponent.csv',
#     'Compound_Gene':                             'Compound_Gene.csv',
#     'Compound_Disease':                          'Compound_Disease.csv',
#     'Compound_Compound':                         'Compound_Compound.csv',
#     'Compound_Phenotype':                        'Compound_Phenotype.csv',
#     'Compound_Pathway':                          'Compound_Pathway.csv',
#     'Compound_SideEffect':                       'Compound_SideEffect.csv',
#     'Compound_Side_Effect':                      'Compound_SideEffect.csv',
#     'Anatomy_Gene':                              'Anatomy_Gene.csv',
#     'Anatomy_Anatomy':                           'Anatomy_Anatomy.csv',
#     'Phenotype_Phenotype':                       'Phenotype_Phenotype.csv',
#     'Pathway_Go':                                'Pathway_Go.csv',
#     'Pathway_Pathway':                           'Pathway_Pathway.csv',
#     'Pathway_Biological_Process':                'Pathway_Biological_Process.csv',
#     'Pathway_Cellular_Component':                'Pathway_Cellular_Component.csv',
#     'Pathway_Molecular_Function':                'Pathway_Molecular_Function.csv',
#     'Go_Go':                                     'Go_Go.csv',
#     'Molecular_Function_Molecular_Function':     'Molecular_Function_Molecular_Function.csv',
#     'Cellular_Component_Cellular_Component':     'Cellular_Component_Cellular_Component.csv',
#     'Biological_Process_Biological_Process':     'Biological_Process_Biological_Process.csv',
#     'Symptom_Disease':                           'Symptom_Disease.csv',
#     # ── TCM relations (present in new download, not in old) ──────────────────
#     # Saved as-is; not processed in downstream cells but preserved for future use
#     # 'TCM_CMM_Compound':                          'TCM_CMM_Compound.csv',
#     # 'TCM_CMM_Disease':                           'TCM_CMM_Disease.csv',
#     # 'TCM_CMM_Gene':                              'TCM_CMM_Gene.csv',
#     # 'TCM_CMM_TCM_Symptom':                       'TCM_CMM_TCM_Symptom.csv',
#     # 'TCM_CMM_TCM_Syndrome':                      'TCM_CMM_TCM_Syndrome.csv',
#     # 'TCM_Prescription_Disease':                  'TCM_Prescription_Disease.csv',
#     # 'TCM_Prescription_TCM_CMM':                  'TCM_Prescription_TCM_CMM.csv',
#     # 'TCM_Prescription_TCM_Symptom':              'TCM_Prescription_TCM_Symptom.csv',
#     # 'TCM_Prescription_TCM_Syndrome':             'TCM_Prescription_TCM_Syndrome.csv',
#     # 'TCM_Symptom_Disease':                       'TCM_Symptom_Disease.csv',
#     # 'TCM_Symptom_Symptom':                       'TCM_Symptom_Symptom.csv',
#     # 'TCM_Symptom_TCM_Syndrome':                  'TCM_Symptom_TCM_Syndrome.csv',
#     # 'TCM_Syndrome_Disease':                      'TCM_Syndrome_Disease.csv',
# }

# # saved = []
# # skipped = []

# # for relation, df_group in E_enriched.groupby('Relation'):
# #     if relation in relation_to_filename:
# #         fname = relation_to_filename[relation]
# #         out   = os.path.join(PROCESSED_TARKG, fname)
# #         # If multiple relation strings map to same file (e.g. Gene_Biological_Process
# #         # and Gene_BiologicalProcess both -> Gene_BiologicalProcess.csv), append rows
# #         if os.path.exists(out):
# #             existing = pd.read_csv(out)
# #             pd.concat([existing, df_group], ignore_index=True).to_csv(out, index=False)
# #             saved.append(f"  {relation:<50s}  {len(df_group):>8,} rows appended -> {fname}")
# #         else:
# #             df_group.to_csv(out, index=False)
# #             saved.append(f"  {relation:<50s}  {len(df_group):>8,} rows  ->  {fname}")
# #     else:
# #         skipped.append(f"  {relation}")

# # print(f"Saved/appended {len(saved)} relation files to: {PROCESSED_TARKG}")
# # for s in saved:
# #     print(s)
# # if skipped:
# #     print(f"\nStill not mapped ({len(skipped)} relations — add to dict above if needed):")
# #     for s in skipped:
# #         print(s)

In [ ]:
# # ── Verify saved files ────────────────────────────────────────────────────────
# import glob as glob_mod
# files = sorted(glob_mod.glob(os.path.join(PROCESSED_TARKG, '*.csv')))
# print(f"Files in Processed_TARKG/: {len(files)}")
# for f in files:
#     rows = sum(1 for _ in open(f)) - 1
#     print(f"  {os.path.basename(f):<55s}  {rows:>7,} rows")

---
## Part 2 · Load reference dictionaries

In [8]:
# ── 2a. PubChem CID -> IUPAC name and SMILES ──────────────────────────────────
Pubchem = pd.read_pickle(PUBCHEM_PKL_PATH)
Pubchem_CID_Smile_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_SMILES']))
Pubchem_IUPAC_CID_Dict = dict(zip(Pubchem['PUBCHEM_COMPOUND_CID'], Pubchem['PUBCHEM_IUPAC_NAME']))
del Pubchem
print("PubChem CID dicts loaded")

PubChem CID dicts loaded


In [9]:
# ── 2b. PubChem crosswalk: DrugBank/ChEBI -> PubChem CID ─────────────────────
pubchem = pd.read_csv(PUBCHEM_DB_PATH)
pubchem_DB    = pubchem[~pubchem['DRUGBANK_ID'].isna()]
pubchem_CHEBI = pubchem[~pubchem['CHEBI_ID'].isna()]
DB2Pub_dict   = dict(zip(pubchem_DB['ID'],          pubchem_DB['DRUGBANK_ID']))   # CID -> DB ID
DB2PC_dict    = dict(zip(pubchem_DB['DRUGBANK_ID'],  pubchem_DB['ID']))            # DB ID -> CID
Chebi2PC_dict = dict(zip(pubchem_CHEBI['CHEBI_ID'], pubchem_CHEBI['ID']))          # ChEBI -> CID
del pubchem
print(f"DrugBank->PubChem: {len(DB2PC_dict):,} | ChEBI->PubChem: {len(Chebi2PC_dict):,}")

/tmp/ipykernel_1739243/2767320013.py:2: DtypeWarning: Columns (1,2) have mixed types. Specify dtype option on import or set low_memory=False.
  pubchem = pd.read_csv(PUBCHEM_DB_PATH)


DrugBank->PubChem: 10,702 | ChEBI->PubChem: 177,680


In [10]:
# ── 2c. PubChem synonym -> CID (case-insensitive) ────────────────────────────
Pubchem_Syn_fil = pd.read_csv(PUBCHEM_SYN_PATH, sep='\t', header=None)
Pubchem_Syn_fil_dict = dict(zip(Pubchem_Syn_fil[1], Pubchem_Syn_fil[0]))
Pubchem_Syn_fil_dict_lower = {k.lower(): v for k, v in Pubchem_Syn_fil_dict.items()}
del Pubchem_Syn_fil
print(f"PubChem synonyms: {len(Pubchem_Syn_fil_dict):,}")

PubChem synonyms: 102,877,691


In [2]:
# ── 2d. ENSEMBL <-> NCBI gene crosswalk ──────────────────────────────────────
ENS_2NCBI = pd.read_csv(ENS2NCBI_PATH)
ENS_2NCBI = ENS_2NCBI[~ENS_2NCBI['name'].isna()]
NCBI_2ENS__dict = dict(zip(ENS_2NCBI['name'], ENS_2NCBI['initial_alias']))
del ENS_2NCBI
print(f"Symbol -> Ensembl: {len(NCBI_2ENS__dict):,}")

Symbol -> Ensembl: 40,940


In [3]:
# ── 2e. NCBI Human gene_info ──────────────────────────────────────────────────
NCBI_ALL_GENE = pd.read_csv(NCBI_HUMAN_PATH, sep='\t')
NCBI_ALL_GENE['ENSEMBLE_ID']   = NCBI_ALL_GENE['Symbol'].map(NCBI_2ENS__dict)
NCBI_ALL_GENEname_dict         = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['description']))
NCBI_ALL_GENEIDname_dict       = dict(zip(NCBI_ALL_GENE['GeneID'],  NCBI_ALL_GENE['Symbol']))
NCBI_Synbol_GENEname_dict      = dict(zip(NCBI_ALL_GENE['Symbol'],  NCBI_ALL_GENE['description']))
# String-keyed version needed for TARKG gene IDs which come as strings
NCBI_ALL_GENEIDname_dict_str   = {str(k): v for k, v in NCBI_ALL_GENEIDname_dict.items()}
print(f"NCBI Human genes: {len(NCBI_Synbol_GENEname_dict):,}")

NCBI Human genes: 193,352


In [4]:
# ── 2f. OMIM -> DOID ──────────────────────────────────────────────────────────
All_ref_omim = pd.read_csv(OMIM_DO_PATH, sep='\t')
All_ref_omim.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
All_ref_omim['xrefs'] = All_ref_omim['xrefs'].str.split(', ')
All_ref_omim = All_ref_omim.explode('xrefs')
# TAR-KG uses 'OMIM:' prefix; OMIMinDO uses 'MIM:' — normalise to OMIM
All_ref_omim['xrefs'] = All_ref_omim['xrefs'].str.replace('MIM', 'OMIM')
OMIM_ID_2_DOID_dict = dict(zip(All_ref_omim['xrefs'], All_ref_omim['id']))
print(f"OMIM -> DOID: {len(OMIM_ID_2_DOID_dict):,}")

OMIM -> DOID: 6,094


In [5]:
# ── 2g. MeSH -> DOID crosswalk ────────────────────────────────────────────────
Mesh2DOID = pd.read_csv(MESH2DOID_PATH, sep='\t')
Mesh2DOID.rename(columns={'ID': 'id', 'LABEL': 'label'}, inplace=True)
Mesh2DOID['xrefs'] = Mesh2DOID['xrefs'].str.split(', ')
Mesh2DOID = Mesh2DOID.explode('xrefs')
Mesh2DOID_dict    = dict(zip(Mesh2DOID['xrefs'],  Mesh2DOID['id']))    # MeSH xref -> DOID
MESH2Name_Dict_1  = dict(zip(Mesh2DOID['xrefs'],  Mesh2DOID['label'])) # MeSH xref -> label
# Remove MESH: prefix for fallback lookups
updated_dict = {k.replace('MESH:', ''): v for k, v in Mesh2DOID_dict.items()}
print(f"MeSH -> DOID: {len(Mesh2DOID_dict):,}")

MeSH -> DOID: 3,687


In [6]:
Mesh2DOID_dict

{'MESH:D006394': 'DOID:0001816',
 'MESH:D008659': 'DOID:0014667',
 'MESH:D000081012': 'DOID:0040091',
 'MESH:D012373': 'DOID:0050052',
 'MESH:D004887': 'DOID:0050061',
 'MESH:C000656784': 'DOID:0050072',
 'MESH:C536166': 'DOID:0050083',
 'MESH:C000656825': 'DOID:0050096',
 'MESH:D004670': 'DOID:0050118',
 'MESH:D051359': 'DOID:0050120',
 'MESH:D010854': 'DOID:13902',
 'MESH:D004604': 'DOID:4976',
 'MESH:D007619': 'DOID:0050144',
 'MESH:D059249': 'DOID:0050147',
 'MESH:D011015': 'DOID:3240',
 'MESH:D054990': 'DOID:0050156',
 'MESH:D018549': 'DOID:0050157',
 'MESH:C562470': 'DOID:0050158',
 'MESH:C562489': 'DOID:0050159',
 'MESH:C571912': 'DOID:0050160',
 'MESH:D004675': 'DOID:0050175',
 'MESH:D004892': 'DOID:0050185',
 'MESH:D015624': 'DOID:0050214',
 'MESH:C536369': 'DOID:0050256',
 'MESH:D058285': 'DOID:0050266',
 'MESH:D014247': 'DOID:0050269',
 'MESH:D060585': 'DOID:0050289',
 'MESH:D060586': 'DOID:0050290',
 'MESH:D000839': 'DOID:0050304',
 'MESH:D003409': 'DOID:0050328',
 'MESH:C5

In [7]:
# ── 2h. MedGen CUI -> preferred name ──────────────────────────────────────────
MedGen_CUID_Source_ID = pd.read_csv(MEDGEN_PATH, sep='\t')
MedGen_CUID_Source_ID = MedGen_CUID_Source_ID.drop_duplicates(subset=['source_id','source'], keep='first')
MedGen_MeshID_name_dict = dict(zip(MedGen_CUID_Source_ID['source_id'], MedGen_CUID_Source_ID['pref_name']))
MedGen_CUID_Source_ID_name_dict = dict(zip(MedGen_CUID_Source_ID['#CUI_or_CN_id'], MedGen_CUID_Source_ID['pref_name']))
MedGen_CUID_Source_ID_name_dict = {k: v.lower() if isinstance(v, str) else v
                                    for k, v in MedGen_CUID_Source_ID_name_dict.items()}
print(f"MedGen CUI -> name: {len(MedGen_CUID_Source_ID_name_dict):,}")

# MedGen HPO mapping: CUI -> SDUI (HPO ID)
MedGen_HPO = pd.read_csv(MEDGEN_HPO_PATH, sep='|')
MedGen_CUID_Source_ID_dict = dict(zip(MedGen_HPO['#CUI'], MedGen_HPO['SDUI']))
print(f"MedGen CUI -> HPO SDUI: {len(MedGen_CUID_Source_ID_dict):,}")

MedGen CUI -> name: 191,887
MedGen CUI -> HPO SDUI: 18,795


In [8]:
# ── 2i. MeSH combined and supplementary ──────────────────────────────────────
MESH = pd.read_csv(MESH_COMB_PATH)
MESH_dict      = dict(zip(MESH['ID'],   MESH['Name']))    # ID -> name
MESH_name_dict = MESH_dict.copy()
name_mesh_dict = dict(zip(MESH['Name'], MESH['ID']))      # name -> ID
name_mesh_dict_lower = {k.lower(): v for k, v in name_mesh_dict.items()}

# MeSH -> PubChem CID (where available in combined file)
MESH['Pubchem_ID'] = MESH['Name'].str.lower().map(Pubchem_Syn_fil_dict_lower)
MESH_pubchem = MESH[~MESH['Pubchem_ID'].isna()].copy()
MESH_pubchem['Pubchem_ID'] = MESH_pubchem['Pubchem_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
MESH_pubchem_dict = dict(zip(MESH_pubchem['ID'], MESH_pubchem['Pubchem_ID']))

Mesh_supp = pd.read_csv(MESH_SUPP_PATH)
Mesh_supp['Pubchem_ID'] = Mesh_supp['Name'].map(Pubchem_Syn_fil_dict)
Mesh_supp_dict = dict(zip(Mesh_supp['ID'], Mesh_supp['Name']))
Mesh_pubchem_supp = Mesh_supp[~Mesh_supp['Pubchem_ID'].isna()].copy()
Mesh_pubchem_supp['Pubchem_ID'] = Mesh_pubchem_supp['Pubchem_ID'].astype(str).str.replace(r'\.0$', '', regex=True)
MESH_pubchem_Supp_dict = dict(zip(Mesh_pubchem_supp['ID'], Mesh_pubchem_supp['Pubchem_ID']))
print(f"MeSH combined: {len(MESH_dict):,} | MeSH->PubChem: {len(MESH_pubchem_dict):,}")

NameError: name 'Pubchem_Syn_fil_dict_lower' is not defined

In [15]:
# ── 2j. Disease Ontology (DO) ─────────────────────────────────────────────────
DO_All_File = pd.read_csv(DO_PATH)
DOID_disname_dict      = dict(zip(DO_All_File['id'],    DO_All_File['label']))
DOID_disAltID_dict     = dict(zip(DO_All_File['id'],    DO_All_File['xrefs']))
DOID_disname_DOID_dict = dict(zip(DO_All_File['label'], DO_All_File['id']))
print(f"DO entries: {len(DOID_disname_DOID_dict):,}")

DO entries: 11,787


In [16]:
# ── 2k. GO terms + secondary ID resolution ────────────────────────────────────
GO = pd.read_csv(GO_PATH)
GO['namespace'] = GO['namespace'].replace({
    'biological_process': 'Biological Process',
    'molecular_function': 'Molecular Function',
    'cellular_component': 'Cellular Component'
})
GO_dict           = dict(zip(GO['id'], GO['name']))
GO_namespace_dict = dict(zip(GO['id'], GO['namespace']))

# Secondary GO IDs -> primary GO ID (for resolving retired identifiers)
GO_SecID = GO[['id','name','namespace','alt_id']].dropna(subset=['alt_id']).copy()
GO_SecID['alt_id'] = GO_SecID['alt_id'].str.split(r'\s*\|\s*')
GO_SecID = GO_SecID.explode('alt_id')
GO_SecID_2_prim_ID_dict = dict(zip(GO_SecID['alt_id'], GO_SecID['id']))
print(f"GO terms: {len(GO_dict):,} | GO secondary IDs: {len(GO_SecID_2_prim_ID_dict):,}")

GO terms: 47,995 | GO secondary IDs: 3,646


In [14]:
# ── 2l. UBERON anatomy, HPO phenotype, Reactome pathways, UniProt ────────────
UBERON = pd.read_csv(UBERON_PATH)
UBERON_dict = dict(zip(UBERON['UBERON ID'], UBERON['Name']))

phenotype = pd.read_csv(HPO_PATH)
phenotype = phenotype[phenotype['id'].str.startswith('HP')]
phenotype_dict        = dict(zip(phenotype['name'], phenotype['id']))  # name -> HPO ID
HPOphenotype_name_dict= dict(zip(phenotype['id'],   phenotype['name']))# HPO ID -> name

def extract_umls(xref):
    """Extract UMLS CUI from an xref string like 'UMLS:C0012345'."""
    if isinstance(xref, str):
        m = re.search(r'UMLS:[A-Za-z0-9]+', xref)
        return m.group(0).replace('UMLS:', '') if m else None
    return None

phenotype['UMLS_ID'] = phenotype['xref'].apply(extract_umls)
phenotype_HPO_UMLS = phenotype[~phenotype['UMLS_ID'].isna()]
UMLS_2_HPO_phenotype_dict = dict(zip(phenotype_HPO_UMLS['UMLS_ID'], phenotype_HPO_UMLS['id']))

pathway_reactome = pd.read_csv(REACTOME_PATH, sep='\t', header=None)
pathway_reactome = pathway_reactome[pathway_reactome[0].str.startswith('R-HSA')]
pathway_reactome_dict = dict(zip(pathway_reactome[0], pathway_reactome[1]))

Uniprot_Recname = pd.read_csv(UNIPROT_REC_PATH)
Uniprot_Recname_dict = dict(zip(Uniprot_Recname['AC'], Uniprot_Recname['RecName']))

print(f"UBERON: {len(UBERON_dict):,} | HPO: {len(HPOphenotype_name_dict):,} | Reactome: {len(pathway_reactome_dict):,} | UniProt: {len(Uniprot_Recname_dict):,}")

UBERON: 15,959 | HPO: 19,483 | Reactome: 2,751 | UniProt: 794,151


---
## Part 3 · Helper functions

In [9]:
PROCESSED_TARKG

'/storage/Arushi/090526_EvoAge/kg_formation/data_collection/tarkg/Processed_TARKG/'

In [10]:
def read_tarkg(filename):
    """Read a per-relation CSV from Processed_TARKG folder."""
    return pd.read_csv(os.path.join(PROCESSED_TARKG, filename))


def resolve_gene_from_ncbigene_id(df, col):
    """
    Resolve NCBI GeneID string column -> canonical NCBI Symbol.
    TAR-KG gene IDs may have 'NCBIGENE:' or 'GENE::' prefixes which are stripped first.
    Only rows where the ID is numeric (i.e. a GeneID) are kept.
    Steps: strip prefix -> keep numeric -> GeneID->Symbol -> Symbol->description -> swap.
    """
    df = df.copy()
    df[col] = df[col].str.replace(r'^NCBIGENE:', '', regex=True)
    df[col] = df[col].str.replace(r'^Gene::', '', regex=True)
    df[col] = df[col].str.replace(r'^NCBIGENE::', '', regex=True)
    df = df[df[col].str.isnumeric()]
    df[col] = df[col].astype(str)
    sym_col    = f'{col}_Symbol'
    detail_col = f'{col}_detail_name'
    df[sym_col]    = df[col].map(NCBI_ALL_GENEIDname_dict_str).fillna(df[col])
    df[detail_col] = df[sym_col].map(NCBI_Synbol_GENEname_dict)
    df = df[~df[detail_col].isna()]
    df[[col, sym_col]] = df[[sym_col, col]]
    df.drop(columns=[sym_col], inplace=True)
    return df


def resolve_disease_head(df, col='Head'):
    """
    Resolve disease name/ID in col -> canonical DOID or MeSH ID.
    Maps via: DO label -> DOID, MeSH name -> MESH ID, MeSH ID -> name,
    MeSH supplementary, then strips MESH: prefix for second-pass lookups.
    Sets {col}_ID_IS based on DOID vs MESH prefix.
    """
    df = df.copy()
    dc = f'{col}_detail_name'
    dc_is = f'{col}_ID_IS'
    df[dc] = (df[col].map(MedGen_CUID_Source_ID_name_dict)
              .fillna(df[col].map(MESH_name_dict))
              .fillna(df[col].map(Mesh_supp_dict))
              .fillna(df[col].map(DOID_disname_dict)))
    df[col] = df[col].str.replace(r'^MESH:', '', regex=True)
    df[dc]  = (df[dc].fillna(df[col].map(MESH_name_dict))
               .fillna(df[col].map(Mesh_supp_dict)))
    df = df[~df[dc].isna()]
    df[dc_is] = np.where(df[col].isna(), None,
            np.where(df[col].str.startswith('DOID'), 'DOID', 'MESH'))
    return df


def select_cols(df):
    """Select only columns that exist in df from DESIRED_COLS."""
    return df[[c for c in DESIRED_COLS if c in df.columns]]


def save(df, filename):
    """Save to OUT_PATH and print confirmation."""
    path = os.path.join(OUT_PATH, filename)
    df.to_csv(path, index=None)
    print(f"Saved {len(df):,} rows -> {path}")


print("Helper functions defined.")

Helper functions defined.


---
## Part 4 · Process and export each relation type

### 1 · Disease_Gene

In [26]:
Disease_Gene = read_tarkg('Disease_Gene.csv')
# Keep only MESH-prefixed disease Head IDs, resolve to DOID where possible
Disease_Gene = Disease_Gene[Disease_Gene['Head'].str.startswith('MESH')]
Disease_Gene['Head'] = Disease_Gene['Head'].map(Mesh2DOID_dict).fillna(Disease_Gene['Head'])
Disease_Gene = resolve_disease_head(Disease_Gene, col='Head')
# Resolve Tail (NCBI GeneID int) -> Symbol
Disease_Gene['Tail'] = pd.to_numeric(Disease_Gene['Tail'], errors='coerce').astype('Int64')
Disease_Gene['Tail_Symbol'] = Disease_Gene['Tail'].map(NCBI_ALL_GENEIDname_dict)
Disease_Gene['Tail_detail_name'] = Disease_Gene['Tail'].map(NCBI_ALL_GENEname_dict)
Disease_Gene = Disease_Gene[~Disease_Gene['Tail_detail_name'].isna()]
Disease_Gene[['Tail','Tail_Symbol']] = Disease_Gene[['Tail_Symbol','Tail']]
Disease_Gene.drop(columns=['Tail_Symbol'], inplace=True)
Disease_Gene['Tail_ID_IS'] = 'NCBI_ID'
Disease_Gene = select_cols(Disease_Gene)
save(Disease_Gene, 'Disease_Gene_TARKG.csv')

/tmp/ipykernel_1739243/2736595898.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(MESH_name_dict))
/tmp/ipykernel_1739243/2736595898.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(Mesh_supp_dict))


Saved 1,303,254 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Disease_Gene_TARKG.csv


In [30]:
Disease_Gene

,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,DOID:0001816,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,angiosarcoma,DNA topoisomerase II alpha,DOID,NCBI_ID,addKG-14368360,addKG,0
1,D003141,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,Communicable Diseases,DNA topoisomerase II alpha,MESH,NCBI_ID,addKG-14303082,addKG,0
2,DOID:225,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,syndrome,DNA topoisomerase II alpha,DOID,NCBI_ID,addKG-10001413,addKG,0
3,DOID:630,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,genetic disease,DNA topoisomerase II alpha,DOID,NCBI_ID,addKG-7589506,addKG,0
4,DOID:10534,Disease_Gene,TOP2A,Disease,associate,Gene,TARKG,stomach cancer,DNA topoisomerase II alpha,DOID,NCBI_ID,addKG-5634124,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2154364,DOID:162,Disease_Gene,TMEM265,Disease,associate,Gene,TARKG,cancer,transmembrane protein 265,DOID,NCBI_ID,addKG-5095209,addKG,0
2155668,D020246,Disease_Gene,SMIM9,Disease,associate,Gene,TARKG,Venous Thrombosis,small integral membrane protein 9,MESH,NCBI_ID,addKG-9771124,addKG,0
2156254,DOID:162,Disease_Gene,OST4,Disease,associate,Gene,TARKG,cancer,"oligosaccharyltransferase complex subunit 4, n...",DOID,NCBI_ID,addKG-13671668,addKG,0
2156767,DOID:162,Disease_Gene,PRAMEF9,Disease,associate,Gene,TARKG,cancer,PRAME family member 9,DOID,NCBI_ID,addKG-7048219,addKG,0


### 2 · Disease_Anatomy

In [27]:
Disease_Anatomy = read_tarkg('Disease_Anatomy.csv')
Disease_Anatomy['Head'] = Disease_Anatomy['Head'].str.replace(r'^Disease::', '', regex=True)
Disease_Anatomy['Head'] = Disease_Anatomy['Head'].map(Mesh2DOID_dict).fillna(Disease_Anatomy['Head'])
Disease_Anatomy['Tail'] = Disease_Anatomy['Tail'].str.replace(r'^Anatomy::', '', regex=True)
Disease_Anatomy = resolve_disease_head(Disease_Anatomy, col='Head')
Disease_Anatomy['Tail_Detail_name'] = Disease_Anatomy['Tail'].map(UBERON_dict)
Disease_Anatomy = Disease_Anatomy[~Disease_Anatomy['Tail'].isna()]
Disease_Anatomy['Tail_ID_IS'] = 'UBERON_ID'
Disease_Anatomy = select_cols(Disease_Anatomy)
save(Disease_Anatomy, 'Disease_Anatomy_TARKG.csv')

Saved 7,042 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Disease_Anatomy_TARKG.csv


/tmp/ipykernel_1739243/2736595898.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(MESH_name_dict))
/tmp/ipykernel_1739243/2736595898.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(Mesh_supp_dict))


### 3 · Gene_Disease

In [60]:
# MedGen_CUID_Source_ID_name_dict.get('C563721')
# MESH_name_dict.get('C563721')
# Mesh_supp_dict.get('C563721')
# Mesh_supp_dict

In [61]:

Gene_Disease = read_tarkg('Gene_Disease.csv')
# Head: NCBI GeneID string -> Symbol
Gene_Disease['Head'] = Gene_Disease['Head'].str.upper().str.replace(r'^GENE::', '', regex=True)
Gene_Disease = resolve_gene_from_ncbigene_id(Gene_Disease, 'Head')
# Tail: disease name -> DOID or MESH ID
Gene_Disease['Tail'] = Gene_Disease['Tail'].str.replace(r'^Disease::', '', regex=True)
Gene_Disease['Tail_ID'] = Gene_Disease['Tail'].map(Mesh2DOID_dict).fillna(Gene_Disease['Tail'])
Gene_Disease[['Tail','Tail_ID']] = Gene_Disease[['Tail_ID','Tail']]

Gene_Disease['Tail'] = Gene_Disease['Tail'].astype(str).str.replace(r'^MESH:', '', regex=True)
Gene_Disease['Tail_detail_name'] = (Gene_Disease['Tail'].map(MedGen_CUID_Source_ID_name_dict)
    .fillna(Gene_Disease['Tail'].map(MESH_name_dict))
    .fillna(Gene_Disease['Tail'].map(Mesh_supp_dict))
    .fillna(Gene_Disease['Tail'].map(DOID_disname_dict)))

Gene_Disease
Gene_Disease['Tail_detail_name'] = Gene_Disease['Tail_detail_name'].fillna(Gene_Disease['Tail'].map(MESH_name_dict))
Gene_Disease = Gene_Disease[~Gene_Disease['Tail_detail_name'].isna()]
Gene_Disease['Tail_ID_IS'] = np.where(Gene_Disease['Tail'].isna(), None,
    np.where(Gene_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MESH'))
Gene_Disease['Head_ID_IS'] = 'NCBI_ID'
Gene_Disease = select_cols(Gene_Disease)
save(Gene_Disease, 'Gene_Disease_TARKG.csv')
Gene_Disease

Saved 57,569 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Disease_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,ACE,Gene_Disease,DOID:175,Gene,GNBR::Md::Gene:Disease,Disease,TARKG,angiotensin I converting enzyme,vascular cancer,NCBI_ID,DOID,DRKG-1712794,DRKG,0
3,MDM2,Gene_Disease,DOID:6193,Gene,GNBR::L::Gene:Disease,Disease,TARKG,MDM2 proto-oncogene,epithelioid sarcoma,NCBI_ID,DOID,DRKG-1750804,DRKG,0
6,FUS,Gene_Disease,DOID:6193,Gene,GNBR::U::Gene:Disease,Disease,TARKG,FUS RNA binding protein,epithelioid sarcoma,NCBI_ID,DOID,DRKG-1729996,DRKG,0
8,NACC1,Gene_Disease,DOID:6193,Gene,GNBR::L::Gene:Disease,Disease,TARKG,nucleus accumbens associated 1,epithelioid sarcoma,NCBI_ID,DOID,DRKG-1701257,DRKG,0
9,MIB1,Gene_Disease,DOID:6193,Gene,GNBR::L::Gene:Disease,Disease,TARKG,MIB E3 ubiquitin protein ligase 1,epithelioid sarcoma,NCBI_ID,DOID,DRKG-1764969,DRKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
325299,B3GALT6,Gene_Disease,C536817,Gene,GNBR::J::Gene:Disease,Disease,TARKG,"beta-1,3-galactosyltransferase 6",Al-Gazali Syndrome,NCBI_ID,MESH,DRKG-1705063,DRKG,0
325303,TRAPPC2,Gene_Disease,C564797,Gene,GNBR::U::Gene:Disease,Disease,TARKG,trafficking protein particle complex subunit 2,"Spondyloepiphyseal Dysplasia Tarda, Autosomal ...",NCBI_ID,MESH,DRKG-1769351,DRKG,0
325305,EFTUD2,Gene_Disease,C535676,Gene,GNBR::U::Gene:Disease,Disease,TARKG,elongation factor Tu GTP binding domain contai...,Richieri Costa Guion-Almeida syndrome,NCBI_ID,MESH,DRKG-1787231,DRKG,0
325314,BHLHA9,Gene_Disease,C563721,Gene,GNBR::U::Gene:Disease,Disease,TARKG,basic helix-loop-helix family member a9,"Syndactyly, Mesoaxial Synostotic, with Phalang...",NCBI_ID,MESH,DRKG-1777236,DRKG,0


In [51]:
# Gene_Disease = read_tarkg('Gene_Disease.csv')
# # Head: NCBI GeneID string -> Symbol
# Gene_Disease['Head'] = Gene_Disease['Head'].str.upper().str.replace(r'^GENE::', '', regex=True)
# Gene_Disease = resolve_gene_from_ncbigene_id(Gene_Disease, 'Head')
# # Tail: disease name -> DOID or MESH ID
# Gene_Disease['Tail'] = Gene_Disease['Tail'].str.replace(r'^Disease::', '', regex=True)
# .map(Mesh2DOID_dict)

# # Gene_Disease =  resolve_disease_head(Gene_Disease, col='Head')#['Tail_detail_name'] = Gene_Disease['Tail'].map()

# # Gene_Disease['Tail_ID'] = (Gene_Disease['Tail'].map(DOID_disname_DOID_dict)
# #     .fillna(Gene_Disease['Tail'].map(name_mesh_dict_lower)))
# # Gene_Disease = Gene_Disease[Gene_Disease['Tail_ID'].astype(str).str.startswith(('DOID','MESH'))]
# # Gene_Disease[['Tail','Tail_ID']] = Gene_Disease[['Tail_ID','Tail']]
# # # Annotate Tail name
# # Gene_Disease['Tail_detail_name'] = (Gene_Disease['Tail'].map(MedGen_CUID_Source_ID_name_dict)
# #     .fillna(Gene_Disease['Tail'].map(MESH_name_dict))
# #     .fillna(Gene_Disease['Tail'].map(Mesh_supp_dict))
# #     .fillna(Gene_Disease['Tail'].map(DOID_disname_dict)))
# # # Gene_Disease['Tail'] = Gene_Disease['Tail'].str.replace(r'^MESH:', '', regex=True)
# # Gene_Disease['Tail'] = Gene_Disease['Tail'].astype(str).str.replace(r'^MESH:', '', regex=True)
# # Gene_Disease
# # Gene_Disease['Tail_detail_name'] = Gene_Disease['Tail_detail_name'].fillna(Gene_Disease['Tail'].map(MESH_name_dict))
# # Gene_Disease = Gene_Disease[~Gene_Disease['Tail_detail_name'].isna()]
# # Gene_Disease['Tail_ID_IS'] = np.where(Gene_Disease['Tail'].isna(), None,
# #     np.where(Gene_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MESH'))
# # Gene_Disease['Head_ID_IS'] = 'NCBI_ID'
# # Gene_Disease = select_cols(Gene_Disease)
# # save(Gene_Disease, 'Gene_Disease_TARKG.csv')
# Gene_Disease

### 4 · Disease_Compound → Disease_ChemicalEntity

In [63]:
Disease_Compound = read_tarkg('Disease_Compound.csv')
# Head disease name -> DOID/MESH ID
Disease_Compound['Head_ID'] = (Disease_Compound['Head'].map(DOID_disname_DOID_dict)
    .fillna(Disease_Compound['Head'].map(name_mesh_dict_lower).fillna(Disease_Compound['Head'])))
Disease_Compound = Disease_Compound[Disease_Compound['Head_ID'].astype(str).str.startswith(('DOID','MESH'))]
Disease_Compound[['Head','Head_ID']] = Disease_Compound[['Head_ID','Head']]
Disease_Compound = resolve_disease_head(Disease_Compound, col='Head')
# Tail compound: strip PUBCHEM.COMPOUND: prefix -> lookup CID
Disease_Compound['Tail'] = Disease_Compound['Tail'].str.replace(r'^PUBCHEM.COMPOUND:', '', regex=True)
Disease_Compound['Tail_ID'] = Disease_Compound['Tail'].map(Pubchem_Syn_fil_dict)
Disease_Compound = Disease_Compound[~Disease_Compound['Tail_ID'].isna()]
Disease_Compound['Tail_ID'] = Disease_Compound['Tail_ID'].astype(str).str.rstrip('.0')
Disease_Compound[['Tail_ID','Tail']] = Disease_Compound[['Tail','Tail_ID']]
Disease_Compound['Tail_IUPAC'] = Disease_Compound['Tail'].map(Pubchem_IUPAC_CID_Dict)
Disease_Compound['Tail_Smiles'] = Disease_Compound['Tail'].map(Pubchem_CID_Smile_Dict)
Disease_Compound['Tail_ID_IS'] = 'Pubchem'
Disease_Compound['Head_ID_IS'] = 'DOID'
Disease_Compound = select_cols(Disease_Compound)
# save(Disease_Compound, 'Disease_Compound_TARKG.csv')

Disease_Compound

/tmp/ipykernel_1739243/2736595898.py:40: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(MESH_name_dict))
/tmp/ipykernel_1739243/2736595898.py:41: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(df[col].map(Mesh_supp_dict))


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_IUPAC,Tail_Smiles,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
35,DOID:2018,Disease_Compound,474,Disease,Mp,Compound,TARKG,hyperinsulinism,"[5-(6-aminopurin-9-yl)-3,4-dihydroxyoxolan-2-y...",CN(C)OP(=O)(O)OCC1C(C(C(O1)N2C=NC3=C(N=CN=C32)...,DOID,Pubchem,PharmKG-349334,PharmKG,1
164,DOID:4483,Disease_Compound,84029,Disease,Mp,Compound,TARKG,rhinitis,"(3R,4S,5S,6R,7R,9R,11R,12R,13S,14R)-6-[(2S,3R,...",CC[C@@H]1[C@@]([C@@H]([C@H](C(=O)[C@@H](C[C@@]...,DOID,Pubchem,PharmKG-314512,PharmKG,1
388,DOID:848,Disease_Compound,3961,Disease,Mp,Compound,TARKG,arthritis,[2-butyl-5-chloro-3-[[4-[2-(2H-tetrazol-5-yl)p...,CCCCC1=NC(=C(N1CC2=CC=C(C=C2)C3=CC=CC=C3C4=NNN...,DOID,Pubchem,PharmKG-337499,PharmKG,1
433,DOID:4029,Disease_Compound,145068,Disease,Mp,Compound,TARKG,gastritis,nitric oxide,[N]=O,DOID,Pubchem,PharmKG-345486,PharmKG,1
464,DOID:437,Disease_Compound,305,Disease,Mp,Compound,TARKG,myasthenia gravis,2-hydroxyethyl(trimethyl)azanium,C[N+](C)(C)CCO,DOID,Pubchem,PharmKG-313374,PharmKG,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3680,DOID:3492,Disease_Compound,2907,Disease,Mp,Compound,TARKG,mixed connective tissue disease,"N,N-bis(2-chloroethyl)-2-oxo-1,3,2lambda5-oxaz...",C1CNP(=O)(OC1)N(CCCl)CCCl,DOID,Pubchem,PharmKG-316563,PharmKG,1
3862,DOID:9182,Disease_Compound,2265,Disease,Mp,Compound,TARKG,pemphigus,6-(3-methyl-5-nitroimidazol-4-yl)sulfanyl-7H-p...,CN1C=NC(=C1SC2=NC=NC3=C2NC=N3)[N+](=O)[O-],DOID,Pubchem,PharmKG-369774,PharmKG,1
3936,DOID:7997,Disease_Compound,6013,Disease,Mp,Compound,TARKG,thyrotoxicosis,"(8R,9S,10R,13S,14S,17S)-17-hydroxy-10,13-dimet...",C[C@]12CC[C@H]3[C@H]([C@@H]1CC[C@@H]2O)CCC4=CC...,DOID,Pubchem,PharmKG-362807,PharmKG,1
5281,DOID:8689,Disease_Compound,5002,Disease,Mp,Compound,TARKG,anorexia nervosa,"2-[2-(4-benzo[b][1,4]benzothiazepin-6-ylpipera...",C1CN(CCN1CCOCCO)C2=NC3=CC=CC=C3SC4=CC=CC=C42,DOID,Pubchem,PharmKG-355541,PharmKG,1


In [64]:
save(Disease_Compound, 'Disease_Compound_TARKG.csv')

Saved 105 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Disease_Compound_TARKG.csv


### 5 · Gene_Phenotype

In [65]:
Gene_Phenotype = read_tarkg('Gene_Phenotype.csv')
Gene_Phenotype['Head'] = Gene_Phenotype['Head'].str.replace(r'^NCBIGENE:', '', regex=True)
Gene_Phenotype = resolve_gene_from_ncbigene_id(Gene_Phenotype, 'Head')
Gene_Phenotype['Tail_detail_name'] = Gene_Phenotype['Tail'].map(HPOphenotype_name_dict)
Gene_Phenotype = Gene_Phenotype[~Gene_Phenotype['Tail_detail_name'].isna()]
Gene_Phenotype['Tail_ID_IS'] = 'HPO_ID'
Gene_Phenotype['Head_ID_IS'] = 'NCBI_ID'
Gene_Phenotype = select_cols(Gene_Phenotype)
save(Gene_Phenotype, 'Gene_Phenotype_TARKG.csv')
Gene_Phenotype

Saved 157,919 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Phenotype_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,TP53,Gene_Phenotype,HP:0030448,Gene,GENE_PHENOTYPE,Phenotype,TARKG,tumor protein p53,Soft tissue sarcoma,NCBI_ID,HPO_ID,OpenBioLink-3740008,OpenBioLink,0
1,BARD1,Gene_Phenotype,HP:0000006,Gene,GENE_PHENOTYPE,Phenotype,TARKG,BRCA1 associated RING domain 1,Autosomal dominant inheritance,NCBI_ID,HPO_ID,OpenBioLink-3222864,OpenBioLink,0
2,CREBBP,Gene_Phenotype,HP:0000006,Gene,GENE_PHENOTYPE,Phenotype,TARKG,CREB binding protein,Autosomal dominant inheritance,NCBI_ID,HPO_ID,OpenBioLink-956482,OpenBioLink,0
3,LITAF,Gene_Phenotype,HP:0000006,Gene,GENE_PHENOTYPE,Phenotype,TARKG,lipopolysaccharide induced TNF factor,Autosomal dominant inheritance,NCBI_ID,HPO_ID,OpenBioLink-4550132,OpenBioLink,0
4,NEDD4L,Gene_Phenotype,HP:0000006,Gene,GENE_PHENOTYPE,Phenotype,TARKG,NEDD4 like E3 ubiquitin protein ligase,Autosomal dominant inheritance,NCBI_ID,HPO_ID,OpenBioLink-1465506,OpenBioLink,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
161219,FARS2,Gene_Phenotype,HP:0025488,Gene,GENE_PHENOTYPE,Phenotype,TARKG,"phenylalanyl-tRNA synthetase 2, mitochondrial",Detrusor sphincter dyssynergia,NCBI_ID,HPO_ID,OpenBioLink-621429,OpenBioLink,0
161220,TMEM231,Gene_Phenotype,HP:0012489,Gene,GENE_PHENOTYPE,Phenotype,TARKG,transmembrane protein 231,Suprasellar arachnoid cyst,NCBI_ID,HPO_ID,OpenBioLink-3922871,OpenBioLink,0
161221,TMEM240,Gene_Phenotype,HP:0007792,Gene,GENE_PHENOTYPE,Phenotype,TARKG,transmembrane protein 240,Microsaccadic pursuit,NCBI_ID,HPO_ID,OpenBioLink-2069906,OpenBioLink,0
161222,ZNF141,Gene_Phenotype,HP:0009374,Gene,GENE_PHENOTYPE,Phenotype,TARKG,zinc finger protein 141,Broad phalanges of the 5th finger,NCBI_ID,HPO_ID,OpenBioLink-3864994,OpenBioLink,0


### 6 & 7 · Gene_Pathway + Protein_Pathway

The raw `Gene_Pathway.csv` contains two node types:
- Numeric Head = NCBI GeneID → Gene_Pathway
- Non-numeric Head = UniProt accession → Protein_Pathway



In [66]:
Gene_Pathway_raw = read_tarkg('Gene_Pathway.csv')
Gene_Pathway_raw['Head'] = Gene_Pathway_raw['Head'].str.replace(r'^NCBIGENE:', '', regex=True)

# ── Gene_Pathway: numeric Head = NCBI GeneID ──────────────────────────────────
Gene_Pathway_1 = Gene_Pathway_raw[Gene_Pathway_raw['Head'].str.isnumeric()].copy()
Gene_Pathway_1 = resolve_gene_from_ncbigene_id(Gene_Pathway_1, 'Head')
Gene_Pathway_1 = Gene_Pathway_1[Gene_Pathway_1['Tail'].str.startswith('REACTOME:')]
Gene_Pathway_1['Tail'] = Gene_Pathway_1['Tail'].str.replace(r'^REACTOME:', '', regex=True)
Gene_Pathway_1['Tail_detail_name'] = Gene_Pathway_1['Tail'].map(pathway_reactome_dict)
# BUG FIX: original had Head_ID_IS='Reactome_ID' and Tail_ID_IS='NCBI_ID' — swapped
Gene_Pathway_1['Head_ID_IS'] = 'NCBI_ID'
Gene_Pathway_1['Tail_ID_IS'] = 'Reactome_ID'
Gene_Pathway_1 = select_cols(Gene_Pathway_1)
save(Gene_Pathway_1, 'Gene_Pathway_TARKG.csv')
Gene_Pathway_1


Saved 104,588 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Pathway_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
38661,AHCTF1,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,AT-hook containing transcription factor 1,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-1613543,OpenBioLink,0
38664,NUP62,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,nucleoporin 62,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-1528822,OpenBioLink,0
38667,NUP37,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,nucleoporin 37,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-3894897,OpenBioLink,0
38670,SEC13,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,"SEC13 homolog, nuclear pore and COPII coat com...",Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-3449725,OpenBioLink,0
38673,SEH1L,Gene_Pathway,R-HSA-1640170,Gene,GENE_PATHWAY,Pathway,TARKG,SEH1 like nucleoporin,Cell Cycle,NCBI_ID,Reactome_ID,OpenBioLink-4063449,OpenBioLink,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
515636,KCNK13,Gene_Pathway,R-HSA-1299287,Gene,GENE_PATHWAY,Pathway,TARKG,potassium two pore domain channel subfamily K ...,Tandem pore domain halothane-inhibited K+ chan...,NCBI_ID,Reactome_ID,OpenBioLink-3067457,OpenBioLink,0
515719,GPCPD1,Gene_Pathway,R-HSA-1483152,Gene,GENE_PATHWAY,Pathway,TARKG,glycerophosphocholine phosphodiesterase 1,Hydrolysis of LPE,NCBI_ID,Reactome_ID,OpenBioLink-3052494,OpenBioLink,0
515724,PLA2G4C,Gene_Pathway,R-HSA-1483152,Gene,GENE_PATHWAY,Pathway,TARKG,phospholipase A2 group IVC,Hydrolysis of LPE,NCBI_ID,Reactome_ID,OpenBioLink-4274897,OpenBioLink,0
515797,MINPP1,Gene_Pathway,R-HSA-1855231,Gene,GENE_PATHWAY,Pathway,TARKG,multiple inositol-polyphosphate phosphatase 1,Synthesis of IPs in the ER lumen,NCBI_ID,Reactome_ID,OpenBioLink-4561526,OpenBioLink,0


In [68]:
 # ── Protein_Pathway: non-numeric Head = UniProt accession ─────────────────────
Protein_Pathway = Gene_Pathway_raw[~Gene_Pathway_raw['Head'].str.isnumeric()].copy()
Protein_Pathway['Head_detail_name'] = Protein_Pathway['Head'].map(Uniprot_Recname_dict)
Protein_Pathway = Protein_Pathway[~Protein_Pathway['Head_detail_name'].isna()]
Protein_Pathway['Tail_detail_name'] = Protein_Pathway['Tail'].map(pathway_reactome_dict)
Protein_Pathway = Protein_Pathway[~Protein_Pathway['Tail_detail_name'].isna()]
Protein_Pathway['Relation']   = 'Protein_Pathway'
Protein_Pathway['Head_type']  = 'Protein'
# BUG FIX: original had Tail_ID_IS='Uniprot_ID' (Head namespace) and Head_ID_IS='Reactome_ID' (Tail namespace)
Protein_Pathway['Head_ID_IS'] = 'Uniprot_ID'
Protein_Pathway['Tail_ID_IS'] = 'Reactome_ID'
Protein_Pathway = select_cols(Protein_Pathway)
save(Protein_Pathway, 'Protein_Pathway_TARKG.csv')
Protein_Pathway


Saved 38,882 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Protein_Pathway_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
44402,Q8WYP5,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Protein ELYS,Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-302944,BioKG,0
44407,Q8NFH4,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Nucleoporin Nup37,Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-110254,BioKG,0
44412,P55735,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Protein SEC13 homolog {ECO:0000305},Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-230542,BioKG,0
44417,Q96EE3,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,Nucleoporin SEH1 {ECO:0000305},Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-263457,BioKG,0
44422,P49792,Protein_Pathway,R-HSA-2467813,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,E3 SUMO-protein ligase RanBP2,Separation of Sister Chromatids,Uniprot_ID,Reactome_ID,BioKG-272727,BioKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
516617,P9WGY9,Protein_Pathway,R-HSA-9639775,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,DNA-directed RNA polymerase subunit beta {ECO:...,Antimicrobial action and antimicrobial resista...,Uniprot_ID,Reactome_ID,BioKG-217143,BioKG,0
516618,P9WGZ1,Protein_Pathway,R-HSA-9639775,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,DNA-directed RNA polymerase subunit alpha {ECO...,Antimicrobial action and antimicrobial resista...,Uniprot_ID,Reactome_ID,BioKG-225026,BioKG,0
516619,P9WGY7,Protein_Pathway,R-HSA-9639775,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,DNA-directed RNA polymerase subunit beta' {ECO...,Antimicrobial action and antimicrobial resista...,Uniprot_ID,Reactome_ID,BioKG-323795,BioKG,0
516620,P9WGY5,Protein_Pathway,R-HSA-9639775,Protein,PROTEIN_PATHWAY_ASSOCIATION,Pathway,TARKG,DNA-directed RNA polymerase subunit omega,Antimicrobial action and antimicrobial resista...,Uniprot_ID,Reactome_ID,BioKG-271089,BioKG,0


### 8 · Gene_Anatomy


In [70]:
Gene_Anatomy = read_tarkg('Gene_Anatomy.csv')
Gene_Anatomy['Head'] = Gene_Anatomy['Head'].str.replace(r'^NCBIGENE:', '', regex=True)
Gene_Anatomy = resolve_gene_from_ncbigene_id(Gene_Anatomy, 'Head')
Gene_Anatomy['Tail_detail_name'] = Gene_Anatomy['Tail'].map(UBERON_dict)
Gene_Anatomy = Gene_Anatomy[~Gene_Anatomy['Tail_detail_name'].isna()]
Gene_Anatomy['Tail_ID_IS'] = 'UBERON_ID'
# BUG FIX: original set Head_type='Gene' then immediately overwrote with 'Protein'
Gene_Anatomy['Head_type'] = 'Gene'
Gene_Anatomy['Tail_type'] = 'Anatomy'
Gene_Anatomy['Relation']  = 'Gene_Anatomy'
Gene_Anatomy['Head_ID_IS'] = 'NCBI_ID'
Gene_Anatomy = select_cols(Gene_Anatomy)
save(Gene_Anatomy, 'Gene_Anatomy_TARKG.csv')
Gene_Anatomy

Saved 1,683,157 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Anatomy_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,TOP2A,Gene_Anatomy,UBERON:0002082,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,DNA topoisomerase II alpha,cardiac ventricle,NCBI_ID,UBERON_ID,OpenBioLink-3737144,OpenBioLink,1
1,TOP2A,Gene_Anatomy,UBERON:0002185,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,DNA topoisomerase II alpha,bronchus,NCBI_ID,UBERON_ID,OpenBioLink-3737152,OpenBioLink,1
2,TOP2A,Gene_Anatomy,UBERON:0002240,Gene,GENE_UNDEREXPRESSED_ANATOMY,Anatomy,TARKG,DNA topoisomerase II alpha,spinal cord,NCBI_ID,UBERON_ID,OpenBioLink-3737454,OpenBioLink,1
3,TOP2A,Gene_Anatomy,UBERON:0000955,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,DNA topoisomerase II alpha,brain,NCBI_ID,UBERON_ID,OpenBioLink-3737108,OpenBioLink,1
4,TOP2A,Gene_Anatomy,UBERON:0000996,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,DNA topoisomerase II alpha,vagina,NCBI_ID,UBERON_ID,OpenBioLink-3737113,OpenBioLink,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3155253,SPANXN1,Gene_Anatomy,UBERON:0000473,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,SPANX family member N1,testis,NCBI_ID,UBERON_ID,OpenBioLink-2459376,OpenBioLink,1
3155255,SPANXN1,Gene_Anatomy,UBERON:0000468,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,SPANX family member N1,multicellular organism,NCBI_ID,UBERON_ID,OpenBioLink-2459375,OpenBioLink,1
3155256,SPANXN5,Gene_Anatomy,UBERON:0000473,Gene,GENE_EXPRESSED_ANATOMY,Anatomy,TARKG,SPANX family member N5,testis,NCBI_ID,UBERON_ID,OpenBioLink-2459486,OpenBioLink,1
3155257,SPANXN5,Gene_Anatomy,UBERON:0000473,Gene,GENE_OVEREXPRESSED_ANATOMY,Anatomy,TARKG,SPANX family member N5,testis,NCBI_ID,UBERON_ID,OpenBioLink-2459487,OpenBioLink,1


### 9 · Go_Go → split by namespace (BiologicalProcess / MolecularFunction / CellularComponent)

In [71]:
GO_GO = read_tarkg('Go_Go.csv')
GO_GO['Head_detail_name'] = GO_GO['Head'].map(GO_dict)
GO_GO['Tail_detail_name'] = GO_GO['Tail'].map(GO_dict)
GO_GO['Head_type'] = GO_GO['Head'].map(GO_namespace_dict)
GO_GO['Tail_type'] = GO_GO['Tail'].map(GO_namespace_dict)
GO_GO = GO_GO[~GO_GO['Head_detail_name'].isna() & ~GO_GO['Tail_detail_name'].isna()]
GO_GO['Tail_ID_IS'] = 'Quick_GO'
GO_GO['Head_ID_IS'] = 'Quick_GO'
GO_GO['Relation'] = GO_GO['Head_type'] + '_' + GO_GO['Tail_type']
GO_GO = select_cols(GO_GO)
print(f"GO_GO total: {len(GO_GO):,}")
print(GO_GO['Relation'].value_counts())

GO_GO[GO_GO['Relation'] == 'Biological Process_Biological Process'].to_csv(
    os.path.join(OUT_PATH, 'BiologicalProcess_BiologicalProcess.csv'), index=None)
GO_GO

GO_GO total: 73,906
Relation
Biological Process_Biological Process    55673
Molecular Function_Molecular Function    12819
Cellular Component_Cellular Component     5414
Name: count, dtype: int64


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,GO:1901349,Biological Process_Biological Process,GO:0071705,Biological Process,IS_A,Biological Process,TARKG,glucosinolate transport,nitrogen compound transport,Quick_GO,Quick_GO,OpenBioLink-4746061,OpenBioLink,0
1,GO:0015707,Biological Process_Biological Process,GO:0071705,Biological Process,IS_A,Biological Process,TARKG,nitrite transport,nitrogen compound transport,Quick_GO,Quick_GO,OpenBioLink-4692020,OpenBioLink,0
2,GO:0015747,Biological Process_Biological Process,GO:0071705,Biological Process,IS_A,Biological Process,TARKG,urate transport,nitrogen compound transport,Quick_GO,Quick_GO,OpenBioLink-4692091,OpenBioLink,0
3,GO:0032218,Biological Process_Biological Process,GO:0071705,Biological Process,IS_A,Biological Process,TARKG,riboflavin transport,nitrogen compound transport,Quick_GO,Quick_GO,OpenBioLink-4701656,OpenBioLink,0
4,GO:0015905,Biological Process_Biological Process,GO:0071705,Biological Process,IS_A,Biological Process,TARKG,bicyclomycin transmembrane transport,nitrogen compound transport,Quick_GO,Quick_GO,OpenBioLink-4692405,OpenBioLink,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
77011,GO:1990676,Cellular Component_Cellular Component,GO:0000138,Cellular Component,PART_OF,Cellular Component,TARKG,Golgi trans cisterna membrane,Golgi trans cisterna,Quick_GO,Quick_GO,OpenBioLink-4756934,OpenBioLink,0
77012,GO:1990695,Cellular Component_Cellular Component,GO:1990676,Cellular Component,PART_OF,Cellular Component,TARKG,obsolete intrinsic component of Golgi trans ci...,Golgi trans cisterna membrane,Quick_GO,Quick_GO,OpenBioLink-4756965,OpenBioLink,0
77013,GO:1990704,Cellular Component_Cellular Component,GO:1990695,Cellular Component,IS_A,Cellular Component,TARKG,obsolete integral component of Golgi trans cis...,obsolete intrinsic component of Golgi trans ci...,Quick_GO,Quick_GO,OpenBioLink-4756980,OpenBioLink,0
77014,GO:1990685,Cellular Component_Cellular Component,GO:1990684,Cellular Component,IS_A,Cellular Component,TARKG,HDL-containing protein-lipid-RNA complex,protein-lipid-RNA complex,Quick_GO,Quick_GO,OpenBioLink-4756942,OpenBioLink,0


In [72]:
GO_GO[GO_GO['Relation'] == 'Molecular Function_Molecular Function'].to_csv(
    os.path.join(OUT_PATH, 'MolecularFunction_MolecularFunction.csv'), index=None)
GO_GO[GO_GO['Relation'] == 'Cellular Component_Cellular Component'].to_csv(
    os.path.join(OUT_PATH, 'CellularComponent_CellularComponent.csv'), index=None)
print(f"Saved 3 GO_GO files -> {OUT_PATH}")

Saved 3 GO_GO files -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/


### 10 · Molecular_Function_Molecular_Function (separate source file)

In [74]:
MOLF = read_tarkg('Molecular_Function_Molecular_Function.csv')
MOLF['Head_detail_name'] = MOLF['Head'].map(GO_dict)
MOLF['Tail_detail_name'] = MOLF['Tail'].map(GO_dict)
MOLF['Head_type'] = MOLF['Head'].map(GO_namespace_dict)
MOLF['Tail_type'] = MOLF['Tail'].map(GO_namespace_dict)
MOLF = MOLF[~MOLF['Head_detail_name'].isna() & ~MOLF['Tail_detail_name'].isna()]
MOLF['Tail_ID_IS'] = 'Quick_GO'
MOLF['Head_ID_IS'] = 'Quick_GO'
MOLF['Relation'] = MOLF['Head_type'] + '_' + MOLF['Tail_type']
MOLF = select_cols(MOLF)
save(MOLF, 'MolecularFunction_MolecularFunction_2.csv')
MOLF

Saved 13,741 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/MolecularFunction_MolecularFunction_2.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,GO:0001227,Molecular Function_Molecular Function,GO:0000981,Molecular Function,is a,Molecular Function,TARKG,"DNA-binding transcription repressor activity, ...","DNA-binding transcription factor activity, RNA...",Quick_GO,Quick_GO,addKG-60399,addKG,0
1,GO:0001228,Molecular Function_Molecular Function,GO:0000981,Molecular Function,is a,Molecular Function,TARKG,"DNA-binding transcription activator activity, ...","DNA-binding transcription factor activity, RNA...",Quick_GO,Quick_GO,addKG-60401,addKG,0
2,GO:0004879,Molecular Function_Molecular Function,GO:0000981,Molecular Function,is a,Molecular Function,TARKG,nuclear receptor activity,"DNA-binding transcription factor activity, RNA...",Quick_GO,Quick_GO,addKG-64088,addKG,0
3,GO:0000217,Molecular Function_Molecular Function,GO:0003677,Molecular Function,is a,Molecular Function,TARKG,DNA secondary structure binding,DNA binding,Quick_GO,Quick_GO,addKG-59710,addKG,0
4,GO:0000497,Molecular Function_Molecular Function,GO:0003677,Molecular Function,is a,Molecular Function,TARKG,DNA template activity,DNA binding,Quick_GO,Quick_GO,addKG-59997,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13736,GO:1990948,Molecular Function_Molecular Function,GO:0055105,Molecular Function,is a,Molecular Function,TARKG,ubiquitin ligase inhibitor activity,ubiquitin-protein transferase inhibitor activity,Quick_GO,Quick_GO,addKG-125282,addKG,0
13737,GO:1990970,Molecular Function_Molecular Function,GO:0070883,Molecular Function,is a,Molecular Function,TARKG,trans-activation response element binding,pre-miRNA binding,Quick_GO,Quick_GO,addKG-125311,addKG,0
13738,GO:2001063,Molecular Function_Molecular Function,GO:0010297,Molecular Function,is a,Molecular Function,TARKG,glucomannan binding,heteropolysaccharide binding,Quick_GO,Quick_GO,addKG-127336,addKG,0
13739,GO:2001072,Molecular Function_Molecular Function,GO:0010297,Molecular Function,is a,Molecular Function,TARKG,galactomannan binding,heteropolysaccharide binding,Quick_GO,Quick_GO,addKG-127345,addKG,0


### 11 · Compound_Gene → ChemicalEntity_Gene


In [75]:
Compound_Gene = read_tarkg('Compound_Gene.csv')
Compound_Gene['Tail'] = Compound_Gene['Tail'].str.replace(r'^Gene:', '', regex=True)
Compound_Gene = resolve_gene_from_ncbigene_id(Compound_Gene, 'Tail')
# Resolve Head compound: DB or MESH prefix -> PubChem CID
Compound_Gene = Compound_Gene[Compound_Gene['Head'].str.startswith(('DB','MESH'))]
Compound_Gene['Head'] = Compound_Gene['Head'].str.replace(r'^MESH:', '', regex=True)
Compound_Gene['Head_detail_name'] = (Compound_Gene['Head'].map(MESH_pubchem_Supp_dict)
    .fillna(Compound_Gene['Head'].map(MESH_pubchem_dict))
    .fillna(Compound_Gene['Head'].map(DB2PC_dict)))
Compound_Gene = Compound_Gene[~Compound_Gene['Head_detail_name'].isna()]
Compound_Gene[['Head','Head_detail_name']] = Compound_Gene[['Head_detail_name','Head']]
Compound_Gene['Head'] = Compound_Gene['Head'].astype(str).str.rstrip('.0')
Compound_Gene['Head_detail_name'] = Compound_Gene['Head'].map(Pubchem_IUPAC_CID_Dict)
Compound_Gene = Compound_Gene[~Compound_Gene['Head_detail_name'].isna()]
Compound_Gene['Tail_ID_IS'] = 'NCBI_ID'
Compound_Gene['Head_type']  = 'ChemicalEntity'
Compound_Gene['Tail_type']  = 'Gene'
Compound_Gene['Relation']   = 'ChemicalEntity_Gene'
Compound_Gene['Head_ID_IS'] = 'Pubchem'
Compound_Gene = select_cols(Compound_Gene)
save(Compound_Gene, 'Compound_Gene.csv')
Compound_Gene

Saved 633,243 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Compound_Gene.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
1,123134263,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,"(3R,6R,9R,15R,18R,21R,24R,30R)-30-ethyl-33-[(E...",DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-11489059,addKG,0
4,145068,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,nitric oxide,DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-13142344,addKG,0
6,149096,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,(2S)-7-fluoro-2-methyl-6-(4-methylpiperazin-1-...,DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-8058208,addKG,0
7,4583,ChemicalEntity_Gene,TOP2A,ChemicalEntity,negative_correlate,Gene,TARKG,7-fluoro-2-methyl-6-(4-methylpiperazin-1-yl)-1...,DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-5539511,addKG,0
8,4583,ChemicalEntity_Gene,TOP2A,ChemicalEntity,associate,Gene,TARKG,7-fluoro-2-methyl-6-(4-methylpiperazin-1-yl)-1...,DNA topoisomerase II alpha,Pubchem,NCBI_ID,addKG-6042942,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4200247,596,ChemicalEntity_Gene,GOT1L1,ChemicalEntity,associate,Gene,TARKG,"4-amino-1-[3,4-dihydroxy-5-(hydroxymethyl)oxol...",glutamic-oxaloacetic transaminase 1 like 1,Pubchem,NCBI_ID,addKG-11519604,addKG,0
4200248,83887,ChemicalEntity_Gene,GOT1L1,ChemicalEntity,associate,Gene,TARKG,(2R)-2-aminobutanedioic acid,glutamic-oxaloacetic transaminase 1 like 1,Pubchem,NCBI_ID,addKG-11519603,addKG,0
4207096,638088,ChemicalEntity_Gene,KRTAP2-1,ChemicalEntity,associate,Gene,TARKG,(E)-stilbene,keratin associated protein 2-1,Pubchem,NCBI_ID,addKG-5693051,addKG,0
4218613,31703,ChemicalEntity_Gene,CGB7,ChemicalEntity,positive_correlate,Gene,TARKG,"(7S,9S)-7-[(2R,4S,5S,6S)-4-amino-5-hydroxy-6-m...",chorionic gonadotropin subunit beta 7,Pubchem,NCBI_ID,addKG-13965733,addKG,0


### 12 · Anatomy_Gene

In [76]:
Anatomy_Gene = read_tarkg('Anatomy_Gene.csv')
Anatomy_Gene['Tail'] = Anatomy_Gene['Tail'].str.replace(r'^Gene::', '', regex=True)
Anatomy_Gene = resolve_gene_from_ncbigene_id(Anatomy_Gene, 'Tail')
Anatomy_Gene['Head'] = Anatomy_Gene['Head'].str.replace(r'^Anatomy::', '', regex=True)
Anatomy_Gene['Head_detail_name'] = Anatomy_Gene['Head'].map(UBERON_dict)
Anatomy_Gene = Anatomy_Gene[~Anatomy_Gene['Head_detail_name'].isna()]
Anatomy_Gene['Tail_ID_IS'] = 'NCBI_ID'
Anatomy_Gene['Head_type']  = 'Anatomy'
Anatomy_Gene['Tail_type']  = 'Gene'
Anatomy_Gene['Relation']   = 'Anatomy_Gene'
Anatomy_Gene['Head_ID_IS'] = 'UBERON_ID'
Anatomy_Gene = select_cols(Anatomy_Gene)
save(Anatomy_Gene, 'Anatomy_Gene.csv')
Anatomy_Gene

Saved 1,446,844 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Anatomy_Gene.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,UBERON:0002185,Anatomy_Gene,TOP2A,Anatomy,AeG,Gene,TARKG,bronchus,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,Hetionet-2130105,Hetionet,0
1,UBERON:0002185,Anatomy_Gene,TOP2A,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,bronchus,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,DRKG-4001307,DRKG,0
2,UBERON:0002240,Anatomy_Gene,TOP2A,Anatomy,AdG,Gene,TARKG,spinal cord,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,Hetionet-803528,Hetionet,0
3,UBERON:0002240,Anatomy_Gene,TOP2A,Anatomy,Hetionet::AdG::Anatomy:Gene,Gene,TARKG,spinal cord,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,DRKG-2674730,DRKG,0
4,UBERON:0000996,Anatomy_Gene,TOP2A,Anatomy,AeG,Gene,TARKG,vagina,DNA topoisomerase II alpha,UBERON_ID,NCBI_ID,Hetionet-1766107,Hetionet,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1446839,UBERON:0002097,Anatomy_Gene,KRTAP20-2,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,skin of body,keratin associated protein 20-2,UBERON_ID,NCBI_ID,DRKG-3997801,DRKG,0
1446840,UBERON:0001003,Anatomy_Gene,KRTAP20-2,Anatomy,AeG,Gene,TARKG,skin epidermis,keratin associated protein 20-2,UBERON_ID,NCBI_ID,Hetionet-1951894,Hetionet,0
1446841,UBERON:0001003,Anatomy_Gene,KRTAP20-2,Anatomy,Hetionet::AeG::Anatomy:Gene,Gene,TARKG,skin epidermis,keratin associated protein 20-2,UBERON_ID,NCBI_ID,DRKG-3823096,DRKG,0
1446842,UBERON:0001037,Anatomy_Gene,KRTAP20-2,Anatomy,AeG,Gene,TARKG,strand of hair,keratin associated protein 20-2,UBERON_ID,NCBI_ID,Hetionet-1935610,Hetionet,0


### 13 · Phenotype_Phenotype

In [77]:
Phenotype_Phenotype = read_tarkg('Phenotype_Phenotype.csv')
Phenotype_Phenotype['Head_detail_name'] = Phenotype_Phenotype['Head'].map(HPOphenotype_name_dict)
Phenotype_Phenotype = Phenotype_Phenotype[~Phenotype_Phenotype['Head_detail_name'].isna()]
Phenotype_Phenotype['Tail_detail_name'] = Phenotype_Phenotype['Tail'].map(HPOphenotype_name_dict)
Phenotype_Phenotype = Phenotype_Phenotype[~Phenotype_Phenotype['Tail_detail_name'].isna()]
Phenotype_Phenotype['Head_ID_IS'] = 'HPO_ID'
Phenotype_Phenotype['Tail_ID_IS'] = 'HPO_ID'
Phenotype_Phenotype['Head_type']  = 'Phenotype'
Phenotype_Phenotype['Tail_type']  = 'Phenotype'
Phenotype_Phenotype['Relation']   = 'Phenotype_Phenotype'
Phenotype_Phenotype = select_cols(Phenotype_Phenotype)
save(Phenotype_Phenotype, 'Phenotype_Phenotype.csv')

Phenotype_Phenotype

Saved 18,416 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Phenotype_Phenotype.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,HP:0002859,Phenotype_Phenotype,HP:0030448,Phenotype,IS_A,Phenotype,TARKG,Rhabdomyosarcoma,Soft tissue sarcoma,HPO_ID,HPO_ID,OpenBioLink-4762814,OpenBioLink,0
1,HP:0010614,Phenotype_Phenotype,HP:0030448,Phenotype,IS_A,Phenotype,TARKG,Fibroma,Soft tissue sarcoma,HPO_ID,HPO_ID,OpenBioLink-4770128,OpenBioLink,0
2,HP:0012218,Phenotype_Phenotype,HP:0030448,Phenotype,IS_A,Phenotype,TARKG,Alveolar soft part sarcoma,Soft tissue sarcoma,HPO_ID,HPO_ID,OpenBioLink-4771847,OpenBioLink,0
3,HP:0012570,Phenotype_Phenotype,HP:0030448,Phenotype,IS_A,Phenotype,TARKG,Synovial sarcoma,Soft tissue sarcoma,HPO_ID,HPO_ID,OpenBioLink-4772222,OpenBioLink,0
4,HP:0100243,Phenotype_Phenotype,HP:0030448,Phenotype,IS_A,Phenotype,TARKG,Leiomyosarcoma,Soft tissue sarcoma,HPO_ID,HPO_ID,OpenBioLink-4776804,OpenBioLink,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18411,HP:3000025,Phenotype_Phenotype,HP:0410016,Phenotype,IS_A,Phenotype,TARKG,Abnormality of ciliary ganglion,Abnormal cranial ganglion morphology,HPO_ID,HPO_ID,OpenBioLink-4778603,OpenBioLink,0
18412,HP:3000031,Phenotype_Phenotype,HP:0410006,Phenotype,IS_A,Phenotype,TARKG,Abnormality of anterior ethmoidal artery,Abnormality of ophthalmic artery,HPO_ID,HPO_ID,OpenBioLink-4778610,OpenBioLink,0
18413,HP:3000032,Phenotype_Phenotype,HP:0410006,Phenotype,IS_A,Phenotype,TARKG,Abnormality of central retinal artery,Abnormality of ophthalmic artery,HPO_ID,HPO_ID,OpenBioLink-4778611,OpenBioLink,0
18414,HP:3000039,Phenotype_Phenotype,HP:0410006,Phenotype,IS_A,Phenotype,TARKG,Abnormality of dorsal nasal artery,Abnormality of ophthalmic artery,HPO_ID,HPO_ID,OpenBioLink-4778624,OpenBioLink,0


### 14 · Pathway_Go → Pathway_GO

In [79]:
# Pathway_Go = read_tarkg('Pathway_Go.csv')
# # Tail is GO ID — resolve to name and namespace
# Pathway_Go['Tail_detail_name'] = Pathway_Go['Tail'].map(GO_dict)
# Pathway_Go['Tail_type']        = Pathway_Go['Tail'].map(GO_namespace_dict)
# Pathway_Go['Tail_ID_IS'] = 'Quick_GO'
# Pathway_Go['Head_ID_IS'] = 'Reactome_ID'
# Pathway_Go = select_cols(Pathway_Go)
# # Note: not saved to CSV in original (left as in-memory table) — saving here for completeness
# save(Pathway_Go, 'Pathway_CellularComponent_TARKG.csv')   #DROPPED RHSA ID's are Reaction id's 
# Pathway_Go

Saved 18,647 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Pathway_CellularComponent_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,R-HSA-1638803,Pathway_Go,GO:0000775,Pathway,PATHWAY_GO_CC,Cellular Component,TARKG,"chromosome, centromeric region",Reactome_ID,Quick_GO,BioKG-2240342,BioKG,0
1,R-HSA-1638821,Pathway_Go,GO:0000775,Pathway,PATHWAY_GO_CC,Cellular Component,TARKG,"chromosome, centromeric region",Reactome_ID,Quick_GO,BioKG-2240343,BioKG,0
2,R-HSA-2467809,Pathway_Go,GO:0000775,Pathway,PATHWAY_GO_CC,Cellular Component,TARKG,"chromosome, centromeric region",Reactome_ID,Quick_GO,BioKG-2240344,BioKG,0
3,R-HSA-2467811,Pathway_Go,GO:0000775,Pathway,PATHWAY_GO_CC,Cellular Component,TARKG,"chromosome, centromeric region",Reactome_ID,Quick_GO,BioKG-2240345,BioKG,0
4,R-HSA-2468287,Pathway_Go,GO:0000775,Pathway,PATHWAY_GO_CC,Cellular Component,TARKG,"chromosome, centromeric region",Reactome_ID,Quick_GO,BioKG-2240346,BioKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
18642,R-HSA-4570573,Pathway_Go,GO:0034202,Pathway,PATHWAY_GO_MF,Molecular Function,TARKG,glycolipid floppase activity,Reactome_ID,Quick_GO,BioKG-2236658,BioKG,0
18643,R-HSA-69242,Pathway_Go,GO:0000084,Pathway,PATHWAY_GO_BP,Biological Process,TARKG,mitotic S phase,Reactome_ID,Quick_GO,BioKG-2233385,BioKG,0
18644,R-HSA-68877,Pathway_Go,GO:0000236,Pathway,PATHWAY_GO_BP,Biological Process,TARKG,mitotic prometaphase,Reactome_ID,Quick_GO,BioKG-2233435,BioKG,0
18645,R-HSA-68875,Pathway_Go,GO:0000088,Pathway,PATHWAY_GO_BP,Biological Process,TARKG,mitotic prophase,Reactome_ID,Quick_GO,BioKG-2233946,BioKG,0


### 15 · Gene_MolecularFunction

In [80]:
Gene_MolecularFunction = read_tarkg('Gene_MolecularFunction.csv')
Gene_MolecularFunction['Head'] = Gene_MolecularFunction['Head'].str.replace(r'^Gene::', '', regex=True)
Gene_MolecularFunction = resolve_gene_from_ncbigene_id(Gene_MolecularFunction, 'Head')
Gene_MolecularFunction['Tail'] = Gene_MolecularFunction['Tail'].str.replace(r'^Molecular Function::', '', regex=True)
Gene_MolecularFunction = Gene_MolecularFunction[Gene_MolecularFunction['Tail'].str.startswith('GO')]
Gene_MolecularFunction['Tail_detail_name'] = Gene_MolecularFunction['Tail'].map(GO_dict)
Gene_MolecularFunction = Gene_MolecularFunction[~Gene_MolecularFunction['Tail_detail_name'].isna()]
Gene_MolecularFunction['Tail_type'] = Gene_MolecularFunction['Tail'].map(GO_namespace_dict)
Gene_MolecularFunction['Head_ID_IS'] = 'NCBI_ID'
Gene_MolecularFunction['Tail_ID_IS'] = 'Quick_GO'
Gene_MolecularFunction['Head_type']  = 'Gene'
Gene_MolecularFunction['Relation']   = 'Gene_' + Gene_MolecularFunction['Tail_type']
Gene_MolecularFunction = select_cols(Gene_MolecularFunction)
save(Gene_MolecularFunction, 'Gene_MolecularFunction_TARKG.csv')
Gene_MolecularFunction

Saved 166,778 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_MolecularFunction_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,CREBBP,Gene_Molecular Function,GO:0000981,Gene,GpMF,Molecular Function,TARKG,CREB binding protein,"DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,Hetionet-1075588,Hetionet,0
3,CREBBP,Gene_Molecular Function,GO:0000981,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,TARKG,CREB binding protein,"DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,DRKG-2946790,DRKG,0
4,NFX1,Gene_Molecular Function,GO:0000981,Gene,GpMF,Molecular Function,TARKG,"nuclear transcription factor, X-box binding 1","DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,Hetionet-1059455,Hetionet,0
8,NFX1,Gene_Molecular Function,GO:0000981,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,TARKG,"nuclear transcription factor, X-box binding 1","DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,DRKG-2930657,DRKG,0
9,HLTF,Gene_Molecular Function,GO:0000981,Gene,GpMF,Molecular Function,TARKG,helicase like transcription factor,"DNA-binding transcription factor activity, RNA...",NCBI_ID,Quick_GO,Hetionet-1095033,Hetionet,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
255435,HS3ST3A1,Gene_Molecular Function,GO:0033872,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,TARKG,heparan sulfate-glucosamine 3-sulfotransferase...,obsolete heparan sulfate 3-sulfotransferase 3 ...,NCBI_ID,Quick_GO,DRKG-2979637,DRKG,0
255436,HS3ST3B1,Gene_Molecular Function,GO:0033872,Gene,GpMF,Molecular Function,TARKG,heparan sulfate-glucosamine 3-sulfotransferase...,obsolete heparan sulfate 3-sulfotransferase 3 ...,NCBI_ID,Quick_GO,Hetionet-1069532,Hetionet,0
255438,HS3ST3B1,Gene_Molecular Function,GO:0033872,Gene,Hetionet::GpMF::Gene:Molecular Function,Molecular Function,TARKG,heparan sulfate-glucosamine 3-sulfotransferase...,obsolete heparan sulfate 3-sulfotransferase 3 ...,NCBI_ID,Quick_GO,DRKG-2940734,DRKG,0
255458,TTLL3,Gene_Molecular Function,GO:0070736,Gene,GpMF,Molecular Function,TARKG,tubulin tyrosine ligase like 3,"protein-glycine ligase activity, initiating",NCBI_ID,Quick_GO,Hetionet-1085210,Hetionet,0


### 16 · Gene_Go → Protein_BiologicalProcess / Protein_CellularComponent / Protein_MolecularFunction

In [19]:
Gene_Go = read_tarkg('Gene_Go.csv')
Gene_Go['Head_detail_name'] = Gene_Go['Head'].map(Uniprot_Recname_dict)
Gene_Go = Gene_Go[~Gene_Go['Head_detail_name'].isna()]
Gene_Go['Tail_detail_name'] = Gene_Go['Tail'].map(GO_dict)
Gene_Go = Gene_Go[~Gene_Go['Tail_detail_name'].isna()]
Gene_Go['Tail_type']  = Gene_Go['Tail'].map(GO_namespace_dict)
Gene_Go['Tail_ID_IS'] = 'Quick_GO'
Gene_Go['Head_ID_IS'] = 'Uniprot_ID'
Gene_Go['Head_type']  = 'Protein'
Gene_Go['Relation']   = Gene_Go['Head_type'] + '_' + Gene_Go['Tail_type']
Gene_Go = select_cols(Gene_Go)

Gene_Go[Gene_Go['Tail_type'] == 'Biological Process'].to_csv(
    os.path.join(OUT_PATH, 'Protein_BiologicalProcess_TARKG.csv'), index=None)
Gene_Go[Gene_Go['Tail_type'] == 'Cellular Component'].to_csv(
    os.path.join(OUT_PATH, 'Protein_CellularComponent_TARKG.csv'), index=None)
Gene_Go[Gene_Go['Tail_type'] == 'Molecular Function'].to_csv(
    os.path.join(OUT_PATH, 'Protein_MolecularFunction.csv'), index=None)
print(f"Saved 3 Protein_GO files -> {OUT_PATH}")
Gene_Go


Saved 3 Protein_GO files -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,Q93YU8,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,Nitrate regulatory gene2 protein {ECO:0000303|...,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-2866080,BioKG,0
1,P0ABD0,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,High-affinity choline transport protein {ECO:0...,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-2586045,BioKG,0
2,P9WPR6,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,Uncharacterized transporter MT0942,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-3077038,BioKG,0
3,P9WPR7,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,Uncharacterized transporter Rv0917,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-3077039,BioKG,0
4,P0ABD2,Protein_Biological Process,GO:0071705,Protein,GO_MF,Biological Process,TARKG,Uncharacterized transporter YeaV,nitrogen compound transport,Uniprot_ID,Quick_GO,BioKG-3078823,BioKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1410719,O82377,Protein_Biological Process,GO:0140301,Protein,GO_MF,Biological Process,TARKG,EMBRYO SURROUNDING FACTOR 1-like protein 6,pollen-stigma interaction,Uniprot_ID,Quick_GO,BioKG-2690307,BioKG,0
1410720,A8MR88,Protein_Biological Process,GO:0140301,Protein,GO_MF,Biological Process,TARKG,EMBRYO SURROUNDING FACTOR 1-like protein 8,pollen-stigma interaction,Uniprot_ID,Quick_GO,BioKG-2690309,BioKG,0
1410721,Q1PDG8,Protein_Biological Process,GO:0140301,Protein,GO_MF,Biological Process,TARKG,EMBRYO SURROUNDING FACTOR 1-like protein 10,pollen-stigma interaction,Uniprot_ID,Quick_GO,BioKG-2690310,BioKG,0
1410722,Q9SHX9,Protein_Biological Process,GO:2000083,Protein,GO_MF,Biological Process,TARKG,Putative F-box protein At1g65770,negative regulation of L-ascorbic acid biosynt...,Uniprot_ID,Quick_GO,BioKG-2696853,BioKG,0


In [18]:
Gene_Go['Tail_type'][0]

'Biological Process'

### 17 · Cellular_Component_Cellular_Component

In [85]:
Cellular_Component_Cellular_Component = read_tarkg('Cellular_Component_Cellular_Component.csv')
Cellular_Component_Cellular_Component['Head_detail_name'] = Cellular_Component_Cellular_Component['Head'].map(GO_dict)
Cellular_Component_Cellular_Component['Tail_detail_name'] = Cellular_Component_Cellular_Component['Tail'].map(GO_dict)
Cellular_Component_Cellular_Component['Head_type'] = Cellular_Component_Cellular_Component['Head'].map(GO_namespace_dict)
Cellular_Component_Cellular_Component['Tail_type'] = Cellular_Component_Cellular_Component['Tail'].map(GO_namespace_dict)
Cellular_Component_Cellular_Component = Cellular_Component_Cellular_Component[
    ~Cellular_Component_Cellular_Component['Head_detail_name'].isna() &
    ~Cellular_Component_Cellular_Component['Tail_detail_name'].isna()]
Cellular_Component_Cellular_Component['Tail_ID_IS'] = 'Quick_GO'
Cellular_Component_Cellular_Component['Head_ID_IS'] = 'Quick_GO'
Cellular_Component_Cellular_Component['Relation'] = (Cellular_Component_Cellular_Component['Head_type'] + '_' +
    Cellular_Component_Cellular_Component['Tail_type'])
Cellular_Component_Cellular_Component = select_cols(Cellular_Component_Cellular_Component)
save(Cellular_Component_Cellular_Component, 'CellularComponent_CellularComponent_2.csv')
Cellular_Component_Cellular_Component

Saved 6,524 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/CellularComponent_CellularComponent_2.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,GO:0031518,Cellular Component_Cellular Component,GO:0000775,Cellular Component,part of,Cellular Component,TARKG,CBF3 complex,"chromosome, centromeric region",Quick_GO,Quick_GO,addKG-131488,addKG,0
1,GO:0061838,Cellular Component_Cellular Component,GO:0000775,Cellular Component,part of,Cellular Component,TARKG,CENP-T-W-S-X complex,"chromosome, centromeric region",Quick_GO,Quick_GO,addKG-136287,addKG,0
2,GO:0034506,Cellular Component_Cellular Component,GO:0000775,Cellular Component,part of,Cellular Component,TARKG,"chromosome, centromeric core domain","chromosome, centromeric region",Quick_GO,Quick_GO,addKG-132411,addKG,0
3,GO:0000779,Cellular Component_Cellular Component,GO:0000775,Cellular Component,is a,Cellular Component,TARKG,"condensed chromosome, centromeric region","chromosome, centromeric region",Quick_GO,Quick_GO,addKG-60117,addKG,0
4,GO:0005721,Cellular Component_Cellular Component,GO:0000775,Cellular Component,part of,Cellular Component,TARKG,pericentric heterochromatin,"chromosome, centromeric region",Quick_GO,Quick_GO,addKG-129043,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6519,GO:1990657,Cellular Component_Cellular Component,GO:1990658,Cellular Component,is a,Cellular Component,TARKG,iNOS-S100A8/A9 complex,transnitrosylase complex,Quick_GO,Quick_GO,addKG-124999,addKG,0
6520,GO:1990675,Cellular Component_Cellular Component,GO:0005797,Cellular Component,part of,Cellular Component,TARKG,Golgi medial cisterna membrane,Golgi medial cisterna,Quick_GO,Quick_GO,addKG-142009,addKG,0
6521,GO:1990676,Cellular Component_Cellular Component,GO:0000138,Cellular Component,part of,Cellular Component,TARKG,Golgi trans cisterna membrane,Golgi trans cisterna,Quick_GO,Quick_GO,addKG-142010,addKG,0
6522,GO:1990685,Cellular Component_Cellular Component,GO:1990684,Cellular Component,is a,Cellular Component,TARKG,HDL-containing protein-lipid-RNA complex,protein-lipid-RNA complex,Quick_GO,Quick_GO,addKG-125027,addKG,0


### 18 · Gene_Compound → Gene_ChemicalEntity

In [86]:
Gene_Compound = read_tarkg('Gene_Compound.csv')
Gene_Compound = Gene_Compound[Gene_Compound['Head'].str.startswith('NCBI')]
Gene_Compound = Gene_Compound[Gene_Compound['Tail'].str.startswith('PUB')]
Gene_Compound['Head'] = Gene_Compound['Head'].str.replace(r'^NCBIGENE:', '', regex=True)
Gene_Compound['Tail'] = Gene_Compound['Tail'].str.replace(r'^PUBCHEM.COMPOUND:', '', regex=True)
Gene_Compound = resolve_gene_from_ncbigene_id(Gene_Compound, 'Head')
Gene_Compound['Tail_detail_name'] = Gene_Compound['Tail'].map(Pubchem_IUPAC_CID_Dict)
Gene_Compound = Gene_Compound[~Gene_Compound['Tail_detail_name'].isna()]
Gene_Compound['Tail_SMILES_name'] = Gene_Compound['Tail'].map(Pubchem_CID_Smile_Dict)
Gene_Compound['Tail_ID_IS'] = 'Pubchem'
Gene_Compound['Head_ID_IS'] = 'NCBI_ID'
Gene_Compound['Tail_type']  = 'ChemicalEntity'
Gene_Compound['Relation']   = Gene_Compound['Head_type'] + '_' + Gene_Compound['Tail_type']
Gene_Compound = select_cols(Gene_Compound)
save(Gene_Compound, 'Gene_Compound.csv')
Gene_Compound

Saved 173,931 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Compound.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_SMILES_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,TOP2A,Gene_ChemicalEntity,4583,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,CC1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)F)C(...,7-fluoro-2-methyl-6-(4-methylpiperazin-1-yl)-1...,NCBI_ID,Pubchem,OpenBioLink-3737050,OpenBioLink,1
1,TOP2A,Gene_ChemicalEntity,149096,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,C[C@H]1COC2=C3N1C=C(C(=O)C3=CC(=C2N4CCN(CC4)C)...,(2S)-7-fluoro-2-methyl-6-(4-methylpiperazin-1-...,NCBI_ID,Pubchem,OpenBioLink-3737010,OpenBioLink,1
2,TOP2A,Gene_ChemicalEntity,5379,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,CC1CN(CCN1)C2=C(C=C3C(=C2OC)N(C=C(C3=O)C(=O)O)...,1-cyclopropyl-6-fluoro-8-methoxy-7-(3-methylpi...,NCBI_ID,Pubchem,OpenBioLink-3737062,OpenBioLink,1
3,TOP2A,Gene_ChemicalEntity,2764,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,C1CC1N2C=C(C(=O)C3=CC(=C(C=C32)N4CCNCC4)F)C(=O)O,1-cyclopropyl-6-fluoro-4-oxo-7-piperazin-1-ylq...,NCBI_ID,Pubchem,OpenBioLink-3737021,OpenBioLink,1
4,TOP2A,Gene_ChemicalEntity,101526,Gene,GENE_DRUG,ChemicalEntity,TARKG,DNA topoisomerase II alpha,COC1=C2C(=CC(=C1N3C[C@@H]4CCCN[C@@H]4C3)F)C(=O...,"7-[(4aS,7aS)-1,2,3,4,4a,5,7,7a-octahydropyrrol...",NCBI_ID,Pubchem,OpenBioLink-3736993,OpenBioLink,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
219206,ATP5MGL,Gene_ChemicalEntity,5957,Gene,GENE_DRUG,ChemicalEntity,TARKG,ATP synthase membrane subunit g like,C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H...,"[[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihy...",NCBI_ID,Pubchem,OpenBioLink-1705174,OpenBioLink,1
219207,ATP5MGL,Gene_ChemicalEntity,6022,Gene,GENE_DRUG,ChemicalEntity,TARKG,ATP synthase membrane subunit g like,C1=NC(=C2C(=N1)N(C=N2)[C@H]3[C@@H]([C@@H]([C@H...,"[(2R,3S,4R,5R)-5-(6-aminopurin-9-yl)-3,4-dihyd...",NCBI_ID,Pubchem,OpenBioLink-1705175,OpenBioLink,1
219208,ATP5MGL,Gene_ChemicalEntity,1003,Gene,GENE_DRUG,ChemicalEntity,TARKG,ATP synthase membrane subunit g like,OP(=O)(O)[O-],dihydrogen phosphate,NCBI_ID,Pubchem,OpenBioLink-1705173,OpenBioLink,1
219209,ATP5MGL,Gene_ChemicalEntity,783,Gene,GENE_DRUG,ChemicalEntity,TARKG,ATP synthase membrane subunit g like,[HH],molecular hydrogen,NCBI_ID,Pubchem,OpenBioLink-1705176,OpenBioLink,1


### 19 & 20 · Gene_Gene → Gene_Gene (NCBI) + Protein_Protein (UniProt)



In [87]:
Gene_Gene_raw = read_tarkg('Gene_Gene.csv')
for col in ['Head','Tail']:
    for pat in [r'^NCBIGENE::',r'^Gene::',r'^NCBIGENE:']:
        Gene_Gene_raw[col] = Gene_Gene_raw[col].str.replace(pat, '', regex=True)

# ── Gene_Gene: both Head and Tail numeric ─────────────────────────────────────
Gene_Gene_1 = Gene_Gene_raw[Gene_Gene_raw['Head'].str.isnumeric()].copy()
Gene_Gene_1 = Gene_Gene_1[Gene_Gene_1['Tail'].str.isnumeric()]
Gene_Gene_1['Head'] = Gene_Gene_1['Head'].astype(str)
Gene_Gene_1['Tail'] = Gene_Gene_1['Tail'].astype(str)
# BUG FIX: original had fillna(Gene_Compound['Head']) — wrong DataFrame
Gene_Gene_1['Head_Symbol'] = Gene_Gene_1['Head'].map(NCBI_ALL_GENEIDname_dict_str).fillna(Gene_Gene_1['Head'])
Gene_Gene_1['Head_detail_name'] = Gene_Gene_1['Head_Symbol'].map(NCBI_Synbol_GENEname_dict)
Gene_Gene_1 = Gene_Gene_1[~Gene_Gene_1['Head_detail_name'].isna()]
Gene_Gene_1[['Head','Head_Symbol']] = Gene_Gene_1[['Head_Symbol','Head']]
Gene_Gene_1.drop(columns=['Head_Symbol'], inplace=True)
Gene_Gene_1['Tail_Symbol'] = Gene_Gene_1['Tail'].map(NCBI_ALL_GENEIDname_dict_str).fillna(Gene_Gene_1['Tail'])
Gene_Gene_1['Tail_detail_name'] = Gene_Gene_1['Tail_Symbol'].map(NCBI_Synbol_GENEname_dict)
Gene_Gene_1 = Gene_Gene_1[~Gene_Gene_1['Tail_detail_name'].isna()]
Gene_Gene_1[['Tail','Tail_Symbol']] = Gene_Gene_1[['Tail_Symbol','Tail']]
Gene_Gene_1.drop(columns=['Tail_Symbol'], inplace=True)
Gene_Gene_1['Tail_ID_IS'] = 'NCBI_ID'
Gene_Gene_1['Head_ID_IS'] = 'NCBI_ID'
Gene_Gene_1['Tail_type']  = 'Gene'
Gene_Gene_1['Relation']   = Gene_Gene_1['Head_type'] + '_' + Gene_Gene_1['Tail_type']
Gene_Gene_1 = select_cols(Gene_Gene_1)
save(Gene_Gene_1, 'Gene_Gene_1_TARKG.csv')
Gene_Gene_1

Saved 6,016,029 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_Gene_1_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,NUP62,Gene_Gene,TOP2A,Gene,GENE_CATALYSIS_GENE,Gene,TARKG,nucleoporin 62,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,OpenBioLink-1528454,OpenBioLink,0
1,NUP62,Gene_Gene,TOP2A,Gene,STRING::CATALYSIS::Gene:Gene,Gene,TARKG,nucleoporin 62,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,DRKG-5143357,DRKG,0
2,NUP37,Gene_Gene,TOP2A,Gene,GENE_CATALYSIS_GENE,Gene,TARKG,nucleoporin 37,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,OpenBioLink-3894487,OpenBioLink,0
3,NUP37,Gene_Gene,TOP2A,Gene,STRING::CATALYSIS::Gene:Gene,Gene,TARKG,nucleoporin 37,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,DRKG-5164526,DRKG,0
17,MDM2,Gene_Gene,TOP2A,Gene,negative_correlate,Gene,TARKG,MDM2 proto-oncogene,DNA topoisomerase II alpha,NCBI_ID,NCBI_ID,addKG-10451133,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8436787,KRTAP27-1,Gene_Gene,KRTAP25-1,Gene,STRING::OTHER::Gene:Gene,Gene,TARKG,keratin associated protein 27-1,keratin associated protein 25-1,NCBI_ID,NCBI_ID,DRKG-5858395,DRKG,0
8436788,KRTAP19-7,Gene_Gene,KRTAP20-2,Gene,STRING::OTHER::Gene:Gene,Gene,TARKG,keratin associated protein 19-7,keratin associated protein 20-2,NCBI_ID,NCBI_ID,DRKG-5078774,DRKG,0
8436789,KRTAP19-5,Gene_Gene,KRTAP20-2,Gene,STRING::OTHER::Gene:Gene,Gene,TARKG,keratin associated protein 19-5,keratin associated protein 20-2,NCBI_ID,NCBI_ID,DRKG-5675099,DRKG,0
8436790,KRTAP19-4,Gene_Gene,KRTAP20-2,Gene,STRING::OTHER::Gene:Gene,Gene,TARKG,keratin associated protein 19-4,keratin associated protein 20-2,NCBI_ID,NCBI_ID,DRKG-4996598,DRKG,0


In [88]:

# ── Protein_Protein: both Head and Tail non-numeric (UniProt accessions) ─────
Gene_Gene_2 = Gene_Gene_raw[~Gene_Gene_raw['Head'].str.isnumeric()].copy()
Gene_Gene_2 = Gene_Gene_2[~Gene_Gene_2['Tail'].str.isnumeric()]
Gene_Gene_2['Head_detail_name'] = Gene_Gene_2['Head'].map(Uniprot_Recname_dict)
Gene_Gene_2 = Gene_Gene_2[~Gene_Gene_2['Head_detail_name'].isna()]
Gene_Gene_2['Tail_detail_name'] = Gene_Gene_2['Tail'].map(Uniprot_Recname_dict)
Gene_Gene_2 = Gene_Gene_2[~Gene_Gene_2['Tail_detail_name'].isna()]
Gene_Gene_2['Tail_ID_IS'] = 'Uniprot_ID'
Gene_Gene_2['Head_ID_IS'] = 'Uniprot_ID'
Gene_Gene_2['Tail_type']  = 'Protein'
Gene_Gene_2['Head_type']  = 'Protein'
Gene_Gene_2['Relation']   = 'Protein_Protein'
Gene_Gene_2 = select_cols(Gene_Gene_2)
save(Gene_Gene_2, 'Protein_Protein_TARKG.csv')
Gene_Gene_2

Saved 1,271,469 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Protein_Protein_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
4,Q9Y4X5,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,E3 ubiquitin-protein ligase ARIH1,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522140,addKG,0
5,Q99728,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,BRCA1-associated RING domain protein 1,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522143,addKG,0
6,Q9NZS9,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,Bifunctional apoptosis regulator,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522144,addKG,0
7,P35226,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,Polycomb complex protein BMI-1,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522148,addKG,0
8,Q7Z569,Protein_Protein,P11388,Protein,ubiquitination,Protein,TARKG,BRCA1-associated protein,DNA topoisomerase 2-alpha,Uniprot_ID,Uniprot_ID,addKG-3522149,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8442564,O31697,Protein_Protein,O31697,Protein,PPI,Protein,TARKG,Uncharacterized protein YkzF,Uncharacterized protein YkzF,Uniprot_ID,Uniprot_ID,BioKG-505186,BioKG,0
8442565,O31802,Protein_Protein,O31802,Protein,PPI,Protein,TARKG,Uncharacterized protein YnzH,Uncharacterized protein YnzH,Uniprot_ID,Uniprot_ID,BioKG-537344,BioKG,0
8442566,P34296,Protein_Protein,P34296,Protein,PPI,Protein,TARKG,Uncharacterized protein C06E1.1,Uncharacterized protein C06E1.1,Uniprot_ID,Uniprot_ID,BioKG-537023,BioKG,0
8442567,Q09651,Protein_Protein,Q09651,Protein,PPI,Protein,TARKG,Coiled-coil domain-containing protein 130 homolog,Coiled-coil domain-containing protein 130 homolog,Uniprot_ID,Uniprot_ID,BioKG-561592,BioKG,0


### 21 · Disease_Disease

In [90]:
Disease_Disease = read_tarkg('Disease_Disease.csv')
Disease_Disease = Disease_Disease[Disease_Disease['Head'].str.startswith(('DOID','D0'), na=False)]
Disease_Disease = Disease_Disease[Disease_Disease['Tail'].str.startswith(('DOID','D0'), na=False)]
Disease_Disease['Head_ID'] = Disease_Disease['Head'].map(updated_dict).fillna(Disease_Disease['Head'])
Disease_Disease['Tail_ID'] = Disease_Disease['Tail'].map(updated_dict).fillna(Disease_Disease['Tail'])
Disease_Disease[['Head','Head_ID']] = Disease_Disease[['Head_ID','Head']]
Disease_Disease[['Tail','Tail_ID']] = Disease_Disease[['Tail_ID','Tail']]
Disease_Disease['Head_detail_name'] = (Disease_Disease['Head'].map(DOID_disname_dict)
    .fillna(Disease_Disease['Head'].map(MESH2Name_Dict_1))
    .fillna(Disease_Disease['Head'].map(MESH_name_dict)))
Disease_Disease['Tail_detail_name'] = (Disease_Disease['Tail'].map(DOID_disname_dict)
    .fillna(Disease_Disease['Tail'].map(MESH2Name_Dict_1))
    .fillna(Disease_Disease['Tail'].map(MESH_name_dict)))
Disease_Disease = Disease_Disease[~Disease_Disease['Head_detail_name'].isna() & ~Disease_Disease['Tail_detail_name'].isna()]
Disease_Disease['Head_ID_IS'] = np.where(Disease_Disease['Head'].isna(), None,
    np.where(Disease_Disease['Head'].str.startswith('DOID'), 'DOID', 'MESH'))
Disease_Disease['Tail_ID_IS'] = np.where(Disease_Disease['Tail'].isna(), None,
    np.where(Disease_Disease['Tail'].str.startswith('DOID'), 'DOID', 'MESH'))
Disease_Disease = select_cols(Disease_Disease)
save(Disease_Disease, 'Disease_Disease_TARKG.csv')
Disease_Disease

Saved 35,083 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Disease_Disease_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,DOID:0001816,Disease_Disease,DOID:175,Disease,is a,Disease,TARKG,angiosarcoma,vascular cancer,DOID,DOID,addKG-1,addKG,0
2,DOID:0001816,Disease_Disease,DOID:175,Disease,IS_A,Disease,TARKG,angiosarcoma,vascular cancer,DOID,DOID,OpenBioLink-33345,OpenBioLink,0
3,DOID:0080189,Disease_Disease,DOID:175,Disease,is a,Disease,TARKG,malignant hemangioma,vascular cancer,DOID,DOID,addKG-3090,addKG,0
4,DOID:0080189,Disease_Disease,DOID:175,Disease,IS_A,Disease,TARKG,malignant hemangioma,vascular cancer,DOID,DOID,OpenBioLink-47188,OpenBioLink,0
5,DOID:3316,Disease_Disease,DOID:175,Disease,is a,Disease,TARKG,perivascular tumor,vascular cancer,DOID,DOID,addKG-12041,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71830,D000071378,Disease_Disease,D005531,Disease,is a,Disease,TARKG,Bunion,"Foot Deformities, Acquired",MESH,MESH,addKG-53826,addKG,0
71831,D050489,Disease_Disease,D000071378,Disease,is a,Disease,TARKG,"Bunion, Tailor's",Bunion,MESH,MESH,addKG-53830,addKG,0
71832,D060437,Disease_Disease,D005548,Disease,is a,Disease,TARKG,Artificial Lens Implant Migration,Foreign-Body Migration,MESH,MESH,addKG-59386,addKG,0
71833,D058736,Disease_Disease,D005548,Disease,is a,Disease,TARKG,Intrauterine Device Migration,Foreign-Body Migration,MESH,MESH,addKG-59387,addKG,0


### 22 · Compound_Disease → ChemicalEntity_Disease

In [91]:
Compound_Disease = read_tarkg('Compound_Disease.csv')
Compound_Disease['Head'] = (Compound_Disease['Head']
    .str.replace(r'^Compound::MESH:', '', regex=True)
    .str.replace(r'MESH:', '', regex=True))
Compound_Disease['Head_ID'] = (Compound_Disease['Head'].map(DB2PC_dict)
    .fillna(Compound_Disease['Head'].map(MESH_pubchem_dict))
    .fillna(Compound_Disease['Head'].map(MESH_pubchem_Supp_dict)))
Compound_Disease['Head_ID'] = Compound_Disease['Head_ID'].astype(str).str.rstrip('.0')
Compound_Disease[['Head','Head_ID']] = Compound_Disease[['Head_ID','Head']]
Compound_Disease = Compound_Disease[~Compound_Disease['Head'].isna() & (Compound_Disease['Head'] != 'nan')]
Compound_Disease['Tail_ID'] = (Compound_Disease['Tail'].map(OMIM_ID_2_DOID_dict)
    .fillna(Compound_Disease['Tail'].map(Mesh2DOID_dict)))
Compound_Disease[['Tail','Tail_ID']] = Compound_Disease[['Tail_ID','Tail']]
Compound_Disease = Compound_Disease[~Compound_Disease['Tail'].isna()]
Compound_Disease['Head_detail_name'] = Compound_Disease['Head'].map(Pubchem_IUPAC_CID_Dict)
Compound_Disease['Head_SMILES_name'] = Compound_Disease['Head'].map(Pubchem_CID_Smile_Dict)
Compound_Disease['Tail_detail_name'] = Compound_Disease['Tail'].map(DOID_disname_dict)
Compound_Disease = Compound_Disease[~Compound_Disease['Head_detail_name'].isna() & ~Compound_Disease['Tail_detail_name'].isna()]
Compound_Disease['Head_ID_IS'] = 'Pubchem'
Compound_Disease['Tail_ID_IS'] = 'DOID'
Compound_Disease['Head_type']  = 'ChemicalEntity'
Compound_Disease['Relation']   = Compound_Disease['Head_type'] + '_' + Compound_Disease['Tail_type']
Compound_Disease = select_cols(Compound_Disease)
save(Compound_Disease, 'Compound_Disease_TARKG.csv')
Compound_Disease

Saved 751,743 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Compound_Disease_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,4946,ChemicalEntity_Disease,DOID:175,ChemicalEntity,treat,Disease,TARKG,1-naphthalen-1-yloxy-3-(propan-2-ylamino)propa...,vascular cancer,Pubchem,DOID,addKG-5605254,addKG,0
1,11349,ChemicalEntity_Disease,DOID:175,ChemicalEntity,cause,Disease,TARKG,3-hydroxy-2-phenylchromen-4-one,vascular cancer,Pubchem,DOID,addKG-11487153,addKG,0
2,57469,ChemicalEntity_Disease,DOID:175,ChemicalEntity,treat,Disease,TARKG,"1-(2-methylpropyl)imidazo[4,5-c]quinolin-4-amine",vascular cancer,Pubchem,DOID,addKG-11392101,addKG,0
5,1170711,ChemicalEntity_Disease,DOID:175,ChemicalEntity,treat,Disease,TARKG,3-[[(1-cyclohexyltetrazol-5-yl)methyl-(thiophe...,vascular cancer,Pubchem,DOID,addKG-14470925,addKG,0
6,28,ChemicalEntity_Disease,DOID:175,ChemicalEntity,treat,Disease,TARKG,"2-aminohexa-2,4-dienedioic acid",vascular cancer,Pubchem,DOID,addKG-13001289,addKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2057373,23948,ChemicalEntity_Disease,DOID:0110251,ChemicalEntity,treat,Disease,TARKG,rhodium,cataract 15 multiple types,Pubchem,DOID,addKG-13167009,addKG,0
2057390,469,ChemicalEntity_Disease,DOID:0110170,ChemicalEntity,associate,Disease,TARKG,2-aminohexanedioic acid,Charcot-Marie-Tooth disease axonal type 2Q,Pubchem,DOID,addKG-8463905,addKG,0
2057392,5955,ChemicalEntity_Disease,DOID:0111494,ChemicalEntity,cause,Disease,TARKG,4-nitro-1-oxidoquinolin-1-ium,combined oxidative phosphorylation deficiency 4,Pubchem,DOID,addKG-12139071,addKG,0
2057405,135398643,ChemicalEntity_Disease,DOID:0080458,ChemicalEntity,cause,Disease,TARKG,"[[(2R,3S,4R,5R)-3,4-dihydroxy-5-(6-oxo-1H-puri...",developmental and epileptic encephalopathy 35,Pubchem,DOID,addKG-14512542,addKG,0


### 23 · Gene_BiologicalProcess

In [92]:
Gene_BiologicalProcess = read_tarkg('Gene_BiologicalProcess.csv')
Gene_BiologicalProcess['Head'] = Gene_BiologicalProcess['Head'].str.replace(r'^Gene::', '', regex=True)
Gene_BiologicalProcess['Tail'] = Gene_BiologicalProcess['Tail'].str.replace(r'^Biological Process::', '', regex=True)
Gene_BiologicalProcess = Gene_BiologicalProcess[~Gene_BiologicalProcess['Tail'].str.isnumeric()]
Gene_BiologicalProcess = resolve_gene_from_ncbigene_id(Gene_BiologicalProcess, 'Head')
Gene_BiologicalProcess['Tail_detail_name'] = Gene_BiologicalProcess['Tail'].map(GO_dict)
Gene_BiologicalProcess = Gene_BiologicalProcess[~Gene_BiologicalProcess['Tail_detail_name'].isna()]
Gene_BiologicalProcess['Tail_type'] = Gene_BiologicalProcess['Tail'].map(GO_namespace_dict)
Gene_BiologicalProcess['Head_ID_IS'] = 'NCBI_ID'
Gene_BiologicalProcess['Tail_ID_IS'] = 'QuickGO'
Gene_BiologicalProcess['Head_type']  = 'Gene'
Gene_BiologicalProcess['Relation']   = 'Gene_' + Gene_BiologicalProcess['Tail_type']
Gene_BiologicalProcess = select_cols(Gene_BiologicalProcess)
save(Gene_BiologicalProcess, 'Gene_BiologicalProcess_TARKG.csv')
Gene_BiologicalProcess

Saved 1,045,588 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Gene_BiologicalProcess_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,AHCTF1,Gene_Biological Process,GO:0071705,Gene,GpBP,Biological Process,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,QuickGO,Hetionet-183676,Hetionet,0
1,AHCTF1,Gene_Biological Process,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,Biological Process,TARKG,AT-hook containing transcription factor 1,nitrogen compound transport,NCBI_ID,QuickGO,DRKG-2054878,DRKG,0
2,NUP62,Gene_Biological Process,GO:0071705,Gene,GpBP,Biological Process,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,QuickGO,Hetionet-532807,Hetionet,0
3,NUP62,Gene_Biological Process,GO:0071705,Gene,Hetionet::GpBP::Gene:Biological Process,Biological Process,TARKG,nucleoporin 62,nitrogen compound transport,NCBI_ID,QuickGO,DRKG-2404009,DRKG,0
4,NUP37,Gene_Biological Process,GO:0071705,Gene,GpBP,Biological Process,TARKG,nucleoporin 37,nitrogen compound transport,NCBI_ID,QuickGO,Hetionet-149653,Hetionet,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1223214,SLC35B4,Gene_Biological Process,GO:1990569,Gene,Hetionet::GpBP::Gene:Biological Process,Biological Process,TARKG,solute carrier family 35 member B4,UDP-N-acetylglucosamine transmembrane transport,NCBI_ID,QuickGO,DRKG-2332823,DRKG,0
1223232,PCYOX1,Gene_Biological Process,GO:0030328,Gene,GpBP,Biological Process,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,QuickGO,Hetionet-295606,Hetionet,0
1223234,PCYOX1,Gene_Biological Process,GO:0030328,Gene,Hetionet::GpBP::Gene:Biological Process,Biological Process,TARKG,prenylcysteine oxidase 1,prenylcysteine catabolic process,NCBI_ID,QuickGO,DRKG-2166808,DRKG,0
1223235,PCYOX1L,Gene_Biological Process,GO:0030328,Gene,GpBP,Biological Process,TARKG,prenylcysteine oxidase 1 like,prenylcysteine catabolic process,NCBI_ID,QuickGO,Hetionet-177161,Hetionet,0


### 24 · Compound_Phenotype → ChemicalEntity_Phenotype

In [ ]:
Compound_Phenotype = read_tarkg('Compound_Phenotype.csv')
Compound_Phenotype = Compound_Phenotype[Compound_Phenotype['Tail'].str.startswith('HP')]
Compound_Phenotype['Head'] = Compound_Phenotype['Head'].str.replace(r'^PUBCHEM.COMPOUND:', '', regex=True)
Compound_Phenotype['Head_detail_name'] = Compound_Phenotype['Head'].map(Pubchem_IUPAC_CID_Dict)
Compound_Phenotype['Head_SMILES_name'] = Compound_Phenotype['Head'].map(Pubchem_CID_Smile_Dict)
Compound_Phenotype['Tail_detail_name'] = Compound_Phenotype['Tail'].map(HPOphenotype_name_dict)
Compound_Phenotype = Compound_Phenotype[~Compound_Phenotype['Head_detail_name'].isna() & ~Compound_Phenotype['Tail_detail_name'].isna()]
Compound_Phenotype['Head_ID_IS'] = 'Pubchem'
Compound_Phenotype['Tail_ID_IS'] = 'HPO_ID'
Compound_Phenotype['Head_type']  = 'ChemicalEntity'
Compound_Phenotype['Relation']   = Compound_Phenotype['Head_type'] + '_' + Compound_Phenotype['Tail_type']
Compound_Phenotype = select_cols(Compound_Phenotype)
save(Compound_Phenotype, 'Compound_Phenotype_TARKG.csv')
Compound_Phenotype

### 25 · Compound_SideEffect → ChemicalEntity_SideEffect


In [93]:
Compound_SideEffect = read_tarkg('Compound_SideEffect.csv')
Compound_SideEffect['Head'] = Compound_SideEffect['Head'].str.replace(r'^Compound::', '', regex=True)
Compound_SideEffect['Tail'] = Compound_SideEffect['Tail'].str.replace(r'^Side Effect::', '', regex=True)
Compound_SideEffect['Head_ID'] = Compound_SideEffect['Head'].map(DB2PC_dict).fillna(Compound_SideEffect['Head'])
Compound_SideEffect['Head_ID'] = Compound_SideEffect['Head_ID'].astype(str).str.rstrip('.0')
Compound_SideEffect[['Head_ID','Head']] = Compound_SideEffect[['Head','Head_ID']]
Compound_SideEffect['Head_detail_name'] = Compound_SideEffect['Head'].map(Pubchem_IUPAC_CID_Dict)
Compound_SideEffect['Head_SMILES_name'] = Compound_SideEffect['Head'].map(Pubchem_CID_Smile_Dict)
# UMLS -> HPO ID -> HPO name (two-step)
Compound_SideEffect['Tail_hpo_id']     = Compound_SideEffect['Tail'].map(UMLS_2_HPO_phenotype_dict)
Compound_SideEffect[['Tail_hpo_id','Tail']] = Compound_SideEffect[['Tail','Tail_hpo_id']]
Compound_SideEffect['Tail_detail_name'] = Compound_SideEffect['Tail'].map(HPOphenotype_name_dict)
Compound_SideEffect = Compound_SideEffect[~Compound_SideEffect['Head_detail_name'].isna() & ~Compound_SideEffect['Tail_detail_name'].isna()]
Compound_SideEffect['Head_ID_IS'] = 'Pubchem'
Compound_SideEffect['Tail_ID_IS'] = 'HPO_ID'
Compound_SideEffect['Head_type']  = 'ChemicalEntity'
Compound_SideEffect['Relation']   = Compound_SideEffect['Head_type'] + '_' + Compound_SideEffect['Tail_type']
Compound_SideEffect = select_cols(Compound_SideEffect)
save(Compound_SideEffect, 'Compound_SideEffect_TARKG.csv')
Compound_SideEffect

Saved 205,651 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Compound_SideEffect_TARKG.csv


### 26 · Anatomy_Anatomy


In [94]:
Anatomy_Anatomy = read_tarkg('Anatomy_Anatomy.csv')
Anatomy_Anatomy['Head_detail_name'] = Anatomy_Anatomy['Head'].map(UBERON_dict)
Anatomy_Anatomy = Anatomy_Anatomy[~Anatomy_Anatomy['Head'].isna()]
Anatomy_Anatomy['Tail_detail_name'] = Anatomy_Anatomy['Tail'].map(UBERON_dict)
Anatomy_Anatomy = Anatomy_Anatomy[~Anatomy_Anatomy['Tail'].isna()]
Anatomy_Anatomy['Head_ID_IS'] = 'UBERON_ID'
Anatomy_Anatomy['Tail_ID_IS'] = 'UBERON_ID'
Anatomy_Anatomy['Relation']   = Anatomy_Anatomy['Head_type'] + '_' + Anatomy_Anatomy['Tail_type']
Anatomy_Anatomy = select_cols(Anatomy_Anatomy)
save(Anatomy_Anatomy, 'Anatomy_Anatomy.csv')
Anatomy_Anatomy

Saved 33,344 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Anatomy_Anatomy.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
0,UBERON:0016446,Anatomy_Anatomy,UBERON:0001037,Anatomy,IS_A,Anatomy,TARKG,hair of head,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-23708,OpenBioLink,0
1,UBERON:0000329,Anatomy_Anatomy,UBERON:0001037,Anatomy,PART_OF,Anatomy,TARKG,hair root,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-4216,OpenBioLink,0
2,UBERON:0002074,Anatomy_Anatomy,UBERON:0001037,Anatomy,PART_OF,Anatomy,TARKG,hair shaft,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-7069,OpenBioLink,0
3,UBERON:0005184,Anatomy_Anatomy,UBERON:0001037,Anatomy,PART_OF,Anatomy,TARKG,hair medulla,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-12972,OpenBioLink,0
4,UBERON:0011901,Anatomy_Anatomy,UBERON:0001037,Anatomy,PART_OF,Anatomy,TARKG,hair matrix,strand of hair,UBERON_ID,UBERON_ID,OpenBioLink-20539,OpenBioLink,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
33339,UBERON:6005177,Anatomy_Anatomy,UBERON:6007232,Anatomy,IS_A,Anatomy,TARKG,insect chaeta,insect eo-type sensillum,UBERON_ID,UBERON_ID,OpenBioLink-33213,OpenBioLink,0
33340,UBERON:6007070,Anatomy_Anatomy,UBERON:6040007,Anatomy,IS_A,Anatomy,TARKG,insect centro-posterior medial synaptic neurop...,insect synaptic neuropil domain,UBERON_ID,UBERON_ID,OpenBioLink-33279,OpenBioLink,0
33341,UBERON:6007070,Anatomy_Anatomy,UBERON:6001925,Anatomy,PART_OF,Anatomy,TARKG,insect centro-posterior medial synaptic neurop...,insect embryonic/larval protocerebrum,UBERON_ID,UBERON_ID,OpenBioLink-33280,OpenBioLink,0
33342,UBERON:6007232,Anatomy_Anatomy,UBERON:6007231,Anatomy,IS_A,Anatomy,TARKG,insect eo-type sensillum,insect external sensillum,UBERON_ID,UBERON_ID,OpenBioLink-33292,OpenBioLink,0


### 27 · Pathway_Pathway

In [95]:
Pathway_Pathway = read_tarkg('Pathway_Pathway.csv')
Pathway_Pathway['Head_detail_name'] = Pathway_Pathway['Head'].map(pathway_reactome_dict)
Pathway_Pathway['Tail_detail_name'] = Pathway_Pathway['Tail'].map(pathway_reactome_dict)
Pathway_Pathway = Pathway_Pathway[~Pathway_Pathway['Head_detail_name'].isna() & ~Pathway_Pathway['Tail_detail_name'].isna()]
Pathway_Pathway['Head_ID_IS'] = 'Reactome_ID'
Pathway_Pathway['Tail_ID_IS'] = 'Reactome_ID'
Pathway_Pathway = select_cols(Pathway_Pathway)
save(Pathway_Pathway, 'Pathway_Pathway_TARKG.csv')
Pathway_Pathway

Saved 2,431 rows -> /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/Pathway_Pathway_TARKG.csv


,Head,Relation,Tail,Head_type,Relation_type,Tail_type,KG_Source,Head_detail_name,Tail_detail_name,Head_ID_IS,Tail_ID_IS,Kg_index,Kg,Change
570,R-HSA-1500620,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Meiosis,Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2215978,BioKG,0
571,R-HSA-73886,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Chromosome Maintenance,Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2216900,BioKG,0
572,R-HSA-69620,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Cell Cycle Checkpoints,Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2217275,BioKG,0
573,R-HSA-69278,Pathway_Pathway,R-HSA-1640170,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,"Cell Cycle, Mitotic",Cell Cycle,Reactome_ID,Reactome_ID,BioKG-2231101,BioKG,0
574,R-HSA-2995410,Pathway_Pathway,R-HSA-68882,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Nuclear Envelope (NE) Reassembly,Mitotic Anaphase,Reactome_ID,Reactome_ID,BioKG-2216670,BioKG,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38371,R-HSA-9664417,Pathway_Pathway,R-HSA-9664407,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Leishmania phagocytosis,Parasite infection,Reactome_ID,Reactome_ID,BioKG-2228201,BioKG,0
38372,R-HSA-9662851,Pathway_Pathway,R-HSA-9664433,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,Anti-inflammatory response favouring Leishmani...,Leishmania parasite growth and survival,Reactome_ID,Reactome_ID,BioKG-2228946,BioKG,0
38373,R-HSA-1257604,Pathway_Pathway,R-HSA-9006925,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,PIP3 activates AKT signaling,Intracellular signaling by second messengers,Reactome_ID,Reactome_ID,BioKG-2229924,BioKG,0
38374,R-HSA-1489509,Pathway_Pathway,R-HSA-9006925,Pathway,HAS_PARENT_PATHWAY,Pathway,TARKG,DAG and IP3 signaling,Intracellular signaling by second messengers,Reactome_ID,Reactome_ID,BioKG-2233190,BioKG,0


---
## Part 5 · Summary — all output files

In [20]:
from glob import glob
import pandas as pd, os

print(f"Output files under: {OUT_PATH}\n")
cols = ["Relation","Head_type","Tail_type","Source","KG_Source","Head_ID_IS","Tail_ID_IS"]
df_list = []
for file_path in sorted(glob(os.path.join(OUT_PATH, '*.csv'))):
    try:
        total_df = pd.read_csv(file_path)
        total_rows = len(total_df)
        temp_df = total_df.head(5)
        available = [c for c in cols if c in temp_df.columns]
        row = {'Filename': os.path.basename(file_path), 'no. of triples': total_rows}
        for c in available:
            row[c] = set(temp_df[c].dropna().tolist())
        df_list.append(pd.DataFrame([row]))
    except Exception as e:
        print(f"Error reading {file_path}: {e}")
combined_df = pd.concat(df_list, ignore_index=True)
print(f"Total files: {len(combined_df)}")
combined_df

Output files under: /storage/Arushi/090526_EvoAge/kg_formation/processed_data/TARKG/

Total files: 30


,Filename,no. of triples,Relation,Head_type,Tail_type,KG_Source,Head_ID_IS,Tail_ID_IS
0,Anatomy_Anatomy.csv,33344,{Anatomy_Anatomy},{Anatomy},{Anatomy},{TARKG},{UBERON_ID},{UBERON_ID}
1,Anatomy_Gene.csv,1446844,{Anatomy_Gene},{Anatomy},{Gene},{TARKG},{UBERON_ID},{NCBI_ID}
2,BiologicalProcess_BiologicalProcess.csv,55673,{Biological Process_Biological Process},{Biological Process},{Biological Process},{TARKG},{Quick_GO},{Quick_GO}
3,CellularComponent_CellularComponent.csv,5414,{Cellular Component_Cellular Component},{Cellular Component},{Cellular Component},{TARKG},{Quick_GO},{Quick_GO}
4,CellularComponent_CellularComponent_2.csv,6524,{Cellular Component_Cellular Component},{Cellular Component},{Cellular Component},{TARKG},{Quick_GO},{Quick_GO}
5,Compound_Disease_TARKG.csv,751743,{ChemicalEntity_Disease},{ChemicalEntity},{Disease},{TARKG},{Pubchem},{DOID}
6,Compound_Gene.csv,633243,{ChemicalEntity_Gene},{ChemicalEntity},{Gene},{TARKG},{Pubchem},{NCBI_ID}
7,Compound_SideEffect_TARKG.csv,205651,{ChemicalEntity_Side Effect},{ChemicalEntity},{Side Effect},{TARKG},{Pubchem},{HPO_ID}
8,Disease_Anatomy_TARKG.csv,7042,{Disease_Anatomy},{Disease},{Anatomy},{TARKG},{DOID},{UBERON_ID}
9,Disease_Compound_TARKG.csv,105,{Disease_Compound},{Disease},{Compound},{TARKG},{DOID},{Pubchem}
